<a href="https://www.kaggle.com/code/jakomina/h-logic?scriptVersionId=366857647" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/jakomina/h-logic?scriptVersionId=316677247" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/jakomina/h-logic?scriptVersionId=312709506" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# New Version 

# H7 Framework - Complete Usage Guide
## JSON Extraction Pipeline + Model Integration

---

## 📋 TABLE OF CONTENTS

1. [Quick Start](#quick-start)
2. [Pipeline Architecture](#pipeline-architecture)
3. [Installation & Setup](#installation--setup)
4. [Using the JSON Extraction Pipeline](#using-the-json-extraction-pipeline)
5. [Integrating with Machine Learning Models](#integrating-with-machine-learning-models)
6. [Advanced Usage](#advanced-usage)
7. [API Reference](#api-reference)

---

## 🚀 Quick Start

### Option 1: Kaggle Notebook (Recommended)

```python
# In Kaggle notebook, run this cell:

# Step 1: Load the pipeline
from h7_json_extraction_pipeline import H7ExtractionPipeline

# Step 2: Initialize
pipeline = H7ExtractionPipeline(
    input_dir="/kaggle/input",
    output_dir="/kaggle/working"
)

# Step 3: Execute
splits, summary = pipeline.run()

# Access the data
train_df = splits['train']
test_df = splits['test']
benchmark_df = splits['benchmark']
```

### Option 2: Local Testing

```python
from h7_json_extraction_pipeline import H7ExtractionPipeline
import json

# Create test data
import os
import numpy as np

os.makedirs("test_input", exist_ok=True)
with open("test_input/data.json", 'w') as f:
    for i in range(100):
        record = {'id': i, 'value': np.random.randn()}
        f.write(json.dumps(record) + '\n')

# Run pipeline
pipeline = H7ExtractionPipeline(
    input_dir="test_input",
    output_dir="test_output"
)

splits, summary = pipeline.run()
print("Train shape:", splits['train'].shape)
print("Test shape:", splits['test'].shape)
```

---

## 🏗️ Pipeline Architecture

```
JSON Input Files
      ↓
[STEP 1] File Discovery
      ↓
[STEP 2] Format Detection (JSON vs NDJSON)
      ↓
[STEP 3] Data Extraction
      ↓
[STEP 4] Consolidation (if multiple files)
      ↓
[STEP 5] Flattening (nested structures → columns)
      ↓
[STEP 6] Cleaning (duplicates, null values)
      ↓
[STEP 7] H7 Enrichment (φ, ψ, DRIFT features)
      ↓
[STEP 8] Train/Test/Benchmark Split (70/20/10)
      ↓
CSV Output Files + Summary
      ↓
Ready for ML Models
```

### Supported JSON Formats

#### Standard JSON (Array)
```json
[
  {"id": 1, "name": "Alice", "score": 95.5},
  {"id": 2, "name": "Bob", "score": 87.3}
]
```

#### Standard JSON (Object with array key)
```json
{
  "data": [
    {"id": 1, "name": "Alice"},
    {"id": 2, "name": "Bob"}
  ]
}
```

#### NDJSON (Newline-Delimited JSON)
```
{"id": 1, "name": "Alice", "score": 95.5}
{"id": 2, "name": "Bob", "score": 87.3}
{"id": 3, "name": "Charlie", "score": 92.1}
```

---

## 🔧 Installation & Setup

### Dependencies

```bash
pip install pandas numpy scipy qiskit qiskit-aer matplotlib
```

### In Kaggle

1. Upload `h7_unified_report.json` to Kaggle Dataset
2. Add dataset to your notebook (Data → Your Datasets)
3. Import in notebook:

```python
import sys
sys.path.append('/kaggle/input/notebooks/jakomina/h-logic/h7_unified_report.json')
from h7_json_extraction_pipeline import H7ExtractionPipeline
```

### Local Machine

```bash
# Clone or copy the files
git clone https://github.com/jakobmina/aquora.git
cd aquora

# Install dependencies
pip install -r requirements.txt

# Functions index
python main.py --help
```

---

## 📊 Using the JSON Extraction Pipeline

### Basic Usage

```python
from pipeline_summary import H7ExtractionPipeline

# Initialize pipeline
pipeline = H7ExtractionPipeline(
    input_dir="/kaggle/input",
    output_dir="/kaggle/working"
)

# Run the entire pipeline
splits, summary = pipeline.run()

# Access splits
train_df = splits['train']
test_df = splits['test']
benchmark_df = splits['benchmark']

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Summary: {summary}")
```

### Working with Individual Steps

```python
from pipeline_summary import (
    JSONFileDiscovery,
    JSONExtractor,
    DataProcessing,
    H7Enrichment
)
import pandas as pd

# Step 1: Find JSON files
json_files = JSONFileDiscovery.find_json_files("/path/to/input")
print(f"Found {len(json_files)} files")

# Step 2: Extract data
dfs = []
for json_file in json_files:
    df = JSONExtractor.extract_json_auto(json_file)
    if df is not None:
        dfs.append(df)

# Step 3: Consolidate
full_df = pd.concat(dfs, ignore_index=True)

# Step 4: Flatten
full_df = DataProcessing.flatten_json_smart(full_df)

# Step 5: Clean
full_df = DataProcessing.clean_dataframe_safe(full_df)

# Step 6: Enrich with H7
full_df = H7Enrichment.add_h7_features(full_df)
full_df = H7Enrichment.add_h7_statistics(full_df)

# Save
full_df.to_csv("output.csv", index=False)
```

### Handling Different JSON Formats

```python
from h7_json_extraction_pipeline import JSONFileDiscovery, JSONExtractor

# Auto-detect format
json_path = "data.json"
format_type = JSONFileDiscovery.detect_json_format(json_path)
print(f"Format: {format_type}")  # 'standard', 'ndjson', or 'unknown'

# Extract data (auto-detects format)
df = JSONExtractor.extract_json_auto(json_path)

# Or manually specify format
if format_type == 'ndjson':
    df = JSONExtractor.extract_ndjson(json_path)
else:
    df = JSONExtractor.extract_standard_json(json_path)
```

---

## 🤖 Integrating with Machine Learning Models

### Sklearn Integration

```python
from h7_json_extraction_pipeline import H7ExtractionPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

# Step 1: Extract and prepare data
pipeline = H7ExtractionPipeline()
splits, summary = pipeline.run()

train_df = splits['train']
test_df = splits['test']

# Step 2: Separate features and target
# (adjust column names based on your data)
X_train = train_df.drop(['id', 'target'], axis=1, errors='ignore')
y_train = train_df['target'] if 'target' in train_df.columns else None

X_test = test_df.drop(['id', 'target'], axis=1, errors='ignore')
y_test = test_df['target'] if 'target' in test_df.columns else None

# Step 3: Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Step 4: Evaluate
score = model.score(X_test, y_test)
print(f"Model accuracy: {score:.4f}")

# Step 5: Feature importance (H7 features)
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 H7-enriched features:")
print(feature_importance.head(10))
```

### XGBoost Integration

```python
from h7_json_extraction_pipeline import H7ExtractionPipeline
import xgboost as xgb

# Extract data
pipeline = H7ExtractionPipeline()
splits, summary = pipeline.run()

train_df = splits['train']
test_df = splits['test']

# Prepare data
X_train = train_df.drop(['id', 'target'], axis=1, errors='ignore')
y_train = train_df['target'] if 'target' in train_df.columns else None

X_test = test_df.drop(['id', 'target'], axis=1, errors='ignore')
y_test = test_df['target'] if 'target' in test_df.columns else None

# Create DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Train model
params = {
    'objective': 'binary:logistic',
    'max_depth': 5,
    'learning_rate': 0.1,
    'random_state': 42
}

model = xgb.train(params, dtrain, num_boost_round=100)

# Predict
y_pred = model.predict(dtest)
```

### Pandas + Numpy Integration

```python
from h7_json_extraction_pipeline import H7ExtractionPipeline
import numpy as np

# Extract data
pipeline = H7ExtractionPipeline()
splits, summary = pipeline.run()

df = splits['train']

# Basic statistics on H7-enriched columns
phi_cols = [c for c in df.columns if '_phi_' in c]
print(f"Phi-scaled columns: {len(phi_cols)}")
print(f"Mean of phi features: {df[phi_cols].mean().mean():.4f}")

# Correlation analysis
correlation = df[phi_cols].corr()
print(f"\nCorrelation matrix shape: {correlation.shape}")

# Outlier detection using H7 features
drift_cols = [c for c in df.columns if '_drift_' in c]
for col in drift_cols:
    mean = df[col].mean()
    std = df[col].std()
    outliers = df[(df[col] > mean + 3*std) | (df[col] < mean - 3*std)]
    print(f"{col}: {len(outliers)} outliers detected")
```

### TensorFlow/Keras Integration

```python
from h7_json_extraction_pipeline import H7ExtractionPipeline
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler
import numpy as np

# Extract data
pipeline = H7ExtractionPipeline()
splits, summary = pipeline.run()

train_df = splits['train']
test_df = splits['test']

# Prepare features
X_train = train_df.drop(['id', 'target'], axis=1, errors='ignore')
y_train = train_df['target'].values if 'target' in train_df.columns else None

X_test = test_df.drop(['id', 'target'], axis=1, errors='ignore')
y_test = test_df['target'].values if 'target' in test_df.columns else None

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Build model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # For binary classification
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train
model.fit(
    X_train_scaled, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Evaluate
loss, accuracy = model.evaluate(X_test_scaled, y_test)
print(f"Test accuracy: {accuracy:.4f}")
```

---

## 🔬 Advanced Usage

### Custom H7 Feature Engineering

```python
from h7_json_extraction_pipeline import H7ExtractionPipeline
import numpy as np
import math

phi = (1 + math.sqrt(5)) / 2

pipeline = H7ExtractionPipeline()
splits, summary = pipeline.run()

df = splits['train']

# Add custom H7 features
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    # Custom: Fibonacci-weighted scaling
    df[f'{col}_fib_scaled'] = df[col] * phi * 2
    
    # Custom: H7 modulo operation
    df[f'{col}_mod7'] = df[col] % 7
    
    # Custom: Interaction with H7 constants
    df[f'{col}_h7_interaction'] = df[col] * phi / 7

df.to_csv("enhanced_data.csv", index=False)
```

### Batch Processing Multiple JSON Sources

```python
from h7_json_extraction_pipeline import (
    JSONFileDiscovery,
    JSONExtractor,
    DataProcessing,
    H7Enrichment
)
import os
import pandas as pd

input_dirs = [
    "/kaggle/input/dataset1",
    "/kaggle/input/dataset2",
    "/kaggle/input/dataset3"
]

all_dfs = []

for input_dir in input_dirs:
    print(f"\nProcessing {os.path.basename(input_dir)}...")
    
    # Find files
    json_files = JSONFileDiscovery.find_json_files(input_dir)
    
    # Extract
    for json_file in json_files:
        df = JSONExtractor.extract_json_auto(json_file)
        if df is not None:
            # Process
            df = DataProcessing.flatten_json_smart(df)
            df = DataProcessing.clean_dataframe_safe(df)
            df = H7Enrichment.add_h7_features(df)
            
            all_dfs.append(df)

# Combine all
combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"\nCombined shape: {combined_df.shape}")

combined_df.to_csv("combined_output.csv", index=False)
```

### Memory-Efficient Processing (Large Files)

```python
from h7_json_extraction_pipeline import JSONExtractor, DataProcessing, H7Enrichment
import pandas as pd

# Process NDJSON in chunks
def process_ndjson_chunks(json_path, chunk_size=1000):
    chunks = []
    
    with open(json_path, 'r', encoding='utf-8') as f:
        chunk_lines = []
        for line in f:
            chunk_lines.append(line.strip())
            
            if len(chunk_lines) >= chunk_size:
                # Write temporary JSON
                import json
                records = [json.loads(l) for l in chunk_lines if l]
                df = pd.DataFrame(records)
                
                # Process chunk
                df = DataProcessing.clean_dataframe_safe(df)
                df = H7Enrichment.add_h7_features(df)
                
                chunks.append(df)
                chunk_lines = []
        
        # Process remaining lines
        if chunk_lines:
            import json
            records = [json.loads(l) for l in chunk_lines if l]
            df = pd.DataFrame(records)
            df = DataProcessing.clean_dataframe_safe(df)
            df = H7Enrichment.add_h7_features(df)
            chunks.append(df)
    
    return pd.concat(chunks, ignore_index=True)

# Usage
large_df = process_ndjson_chunks("large_file.json", chunk_size=5000)
print(f"Processed {len(large_df)} records")
```

---

## 📚 API Reference

### H7ExtractionPipeline

```python
class H7ExtractionPipeline:
    def __init__(self, 
                 input_dir: str = "/kaggle/input",
                 output_dir: str = "/kaggle/working"):
        """Initialize pipeline with input/output directories."""
        
    def run(self) -> Tuple[Dict[str, pd.DataFrame], Dict]:
        """
        Execute full pipeline.
        
        Returns:
            splits: Dict with keys 'train', 'test', 'benchmark'
            summary: Dict with pipeline metadata
        """
```

### JSONFileDiscovery

```python
class JSONFileDiscovery:
    @staticmethod
    def find_json_files(input_dir: str) -> List[str]:
        """Find all JSON files in directory."""
        
    @staticmethod
    def detect_json_format(json_path: str) -> str:
        """Detect format: 'standard', 'ndjson', 'unknown'."""
        
    @staticmethod
    def get_file_size_mb(json_path: str) -> float:
        """Get file size in megabytes."""
```

### JSONExtractor

```python
class JSONExtractor:
    @staticmethod
    def extract_ndjson(json_path: str) -> pd.DataFrame:
        """Extract from NDJSON format."""
        
    @staticmethod
    def extract_standard_json(json_path: str) -> pd.DataFrame:
        """Extract from standard JSON format."""
        
    @staticmethod
    def extract_json_auto(json_path: str) -> Optional[pd.DataFrame]:
        """Auto-detect and extract."""
```

### DataProcessing

```python
class DataProcessing:
    @staticmethod
    def flatten_json_smart(df: pd.DataFrame) -> pd.DataFrame:
        """Flatten nested JSON structures."""
        
    @staticmethod
    def clean_dataframe_safe(df: pd.DataFrame) -> pd.DataFrame:
        """Clean data safely."""
```

### H7Enrichment

```python
class H7Enrichment:
    @staticmethod
    def add_h7_features(df: pd.DataFrame) -> pd.DataFrame:
        """Add H7 φ-scaled features."""
        
    @staticmethod
    def add_h7_statistics(df: pd.DataFrame) -> pd.DataFrame:
        """Add statistical features."""
```

---

## 💡 Best Practices

### 1. Always Check Extracted Data

```python
splits, summary = pipeline.run()
print(splits['train'].head())
print(splits['train'].info())
print(splits['train'].describe())
```

### 2. Handle Missing Values

```python
# Check missing values
print(df.isnull().sum())

# Drop rows with too many missing values
df = df.dropna(thresh=len(df) * 0.7)

# Fill missing numeric values
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
```

### 3. Validate H7 Enrichment

```python
# Check H7 features were added
h7_cols = [c for c in df.columns if any(x in c for x in ['phi', 'psi', 'drift'])]
print(f"H7-enriched features: {len(h7_cols)}")

# Verify numerical values
print(df[h7_cols].describe())
```

### 4. Monitor Pipeline Progress

```python
# Capture output
splits, summary = pipeline.run()

print("\nPipeline Summary:")
print(f"  Input files: {summary['input_json_files']}")
print(f"  Total samples: {summary['total_samples']}")
print(f"  Total features: {summary['total_features']}")
print(f"  Train/test/benchmark ratio: {summary['train_samples']}/{summary['test_samples']}/{summary['benchmark_samples']}")
```

---

## 🎯 Common Use Cases

### Kaggle Competition Setup

```python
# In Kaggle notebook
from h7_json_extraction_pipeline import H7ExtractionPipeline
import xgboost as xgb

pipeline = H7ExtractionPipeline()
splits, summary = pipeline.run()

# Prepare submission data
train_df = splits['train']
test_df = splits['test']

X_train = train_df.drop('target', axis=1)
y_train = train_df['target']
X_test = test_df.drop('target', axis=1, errors='ignore')

# Train model
model = xgb.XGBClassifier(random_state=42)
model.fit(X_train, y_train)

# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': model.predict(X_test)
})

submission.to_csv('submission.csv', index=False)
```

---

## 🐛 Troubleshooting

### "No JSON files found"
- Check input directory path
- Ensure files have .json extension

### "Unknown JSON format"
- Verify JSON is valid: `python -m json.tool file.json`
- Try alternative format extraction manually

### "Memory error with large files"
- Use chunk processing (see Advanced Usage)
- Reduce number of H7 features

### Missing H7 features
- Check if columns are numeric
- Verify enrichment ran successfully

---

## 📞 Support

For issues or questions, refer to:
- H7 Framework documentation
- Kaggle notebook examples
- GitHub discussions

---

**Last Updated:** May 2026
**Version:** 1.0
**Stability:** Production Ready ✓

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
! pip install matplotlib qiskit qiskit-aer torch
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import math
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 58.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 86.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.0 MB/s eta 0:00:00
/kaggle/input/datasets/jakomina/cl1-dataet-v1/datos_experimento1.json
/kaggle/input/notebooks/jakomina/h-logic/__huggingface_repos__.json
/kaggle/input/notebooks/jakomina/h-logic/__notebook__.ipynb
/kaggle/input/notebooks/jakomina/h-logic/__output__.json
/kaggle/input/notebooks/jakomina/h-logic/__results__.html
/kaggle/input/notebooks/jakomina/h-logic/benchmark.csv
/kaggle/input/notebooks/jakomina/h-logic/custom.css
/kaggle/input/notebooks/jakomina/h-logic/h7_unified_report.json
/kaggle/input/notebooks/jakomina/h-logic/pipeline_summary.json
/kaggle/input/notebooks/jakomina/h-logic/submission.csv
/kaggle/input/notebooks/jakomina/h-logic/test.csv
/kaggle/in

In [3]:
"""
COVARIANZA DESDE ASIMETRÍA CUÁNTICA (Single-Pass Deduction)
============================================================
Jako's H7 Framework Extension - VERSIÓN SINCRONIZADA (ajustes de consistencia)

Convención de bits ÚNICA en todo el archivo (la de Qiskit, little-endian):
    índice i = q0 + 2·q1 + 4·q2        ket impreso |q2 q1 q0⟩
    bit(i, k) = valor del qubit k en el estado de base i

Resultado central (verificado para n = 0..1000, error ~1e-16):
    Toda la distribución depende de UN escalar
        s = sin(ION)·sin(O_n) − cos(ION)·sin(I_n)·cos(O_n)   (= ⟨Z_q0⟩ tras las rotaciones)
    y la matriz de covarianza es exactamente
        Σ(s) = [[1/4 − s²/16, s/8, s/8],
                [s/8,         1/4, 0  ],
                [s/8,         0,   1/4]]
"""

import os
import json
import warnings
from enum import Enum
from dataclasses import dataclass
from typing import Dict, Tuple

import numpy as np
from scipy.linalg import eigh

warnings.filterwarnings('ignore')

# ============================================================================
# CONSTANTES H7
# ============================================================================
SEED = 7        # semilla fija → corridas reproducibles
N_FIJO = None   # pon un entero para fijar n (p.ej. 1 → O_n = 0.3624, el valor histórico)

rng = np.random.default_rng(SEED)
n = int(N_FIJO) if N_FIJO is not None else int(rng.integers(0, 1001))

pi, phi = np.pi, (1 + np.sqrt(5)) / 2
i_n, o_n = np.cos(pi * n), np.cos(pi * phi * n)  # Residual H7
ion = i_n * o_n
p, q = ion + o_n, ion - o_n                       # Parity normalizer
O_n_integrity = o_n                               # antes se usaba sin definirse (NameError)
DRIFT_072 = 7 - 2 * np.pi                         # Binary cost function mapping

NUM_QUBITS = 3
DIM = 2 ** NUM_QUBITS
SHOTS = 1000

print(f"[INIT] n = {n}  (SEED = {SEED}, N_FIJO = {N_FIJO})")
print(f"[INIT] Golden Ratio φ = {phi:.6f}")
print(f"[INIT] I_n = {i_n:.6f}, O_n = {o_n:.6f}, ION = {ion:.6f}, p = {p:.6f}, q = {q:.6f}")
print(f"[INIT] O_n_integrity = {O_n_integrity:.6f}")
print(f"[INIT] DRIFT_072 = {DRIFT_072:.6f}")


def bit(i: int, k: int) -> int:
    """Valor del qubit k en el índice de base i (convención Qiskit)."""
    return (i >> k) & 1


def ket(i: int, nq: int = NUM_QUBITS) -> str:
    """Etiqueta |q_{nq-1} ... q1 q0⟩ (mismo orden que imprime Qiskit)."""
    return format(i, f'0{nq}b')


# ============================================================================
# CIRCUITO: UNA SOLA DEFINICIÓN (fuente única para numpy, Qiskit y descripción)
# ============================================================================
# (compuerta, qubits, ángulo)
CIRCUITO = [
    ("h",     (0,), None),
    ("h",     (1,), None),
    ("h",     (2,), None),
    ("rz",    (0,), ion),
    ("ry",    (0,), i_n),
    ("rx",    (0,), o_n),
    ("ccx",   (2, 0, 1), None),   # controles q2, q0 → objetivo q1
    ("cswap", (1, 2, 0), None),   # control q1 → intercambia q2 ↔ q0
    ("ccx",   (0, 1, 2), None),   # controles q0, q1 → objetivo q2
]


def describir(circuito) -> str:
    partes = ["H⊗³"]
    rot = [f"{g.upper()}({t:+.4f})" for g, qs, t in circuito if g in ("rz", "ry", "rx")]
    partes.append("·".join(rot) + " en q0")
    for g, qs, _ in circuito:
        if g == "ccx":
            partes.append(f"CCX(q{qs[0]},q{qs[1]}→q{qs[2]})")
        elif g == "cswap":
            partes.append(f"CSWAP(q{qs[0]}: q{qs[1]}↔q{qs[2]})")
    return " → ".join(partes)


# --- Simulador numpy (siempre disponible; no depende de Qiskit) -------------

def _matriz_1q(nombre: str, theta) -> np.ndarray:
    if nombre == "h":
        return np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    if nombre == "rz":
        return np.diag([np.exp(-1j * theta / 2), np.exp(1j * theta / 2)])
    if nombre == "ry":
        return np.array([[c, -s], [s, c]], dtype=complex)
    if nombre == "rx":
        return np.array([[c, -1j * s], [-1j * s, c]], dtype=complex)
    raise ValueError(f"Compuerta de 1 qubit desconocida: {nombre}")


def _permutacion(nombre: str, qs, dim: int) -> np.ndarray:
    """Mapa de índices i → f(i) de una compuerta reversible clásica."""
    f = np.arange(dim)
    for i in range(dim):
        if nombre == "ccx":
            c1, c2, t = qs
            if bit(i, c1) and bit(i, c2):
                f[i] = i ^ (1 << t)
        elif nombre == "cswap":
            c, a, b = qs
            if bit(i, c) and bit(i, a) != bit(i, b):
                f[i] = i ^ (1 << a) ^ (1 << b)
    return f


def simular_numpy(circuito, nq: int = NUM_QUBITS):
    """
    Devuelve (statevector, permutación total de las compuertas reversibles).
    La permutación total es interpretable porque en este circuito todas las
    rotaciones van ANTES de CCX/CSWAP.
    """
    dim = 2 ** nq
    psi = np.zeros(dim, dtype=complex)
    psi[0] = 1.0
    perm_total = np.arange(dim)
    for nombre, qs, theta in circuito:
        if nombre in ("ccx", "cswap"):
            f = _permutacion(nombre, qs, dim)
            nuevo = np.empty_like(psi)
            nuevo[f] = psi
            psi = nuevo
            perm_total = f[perm_total]
        else:
            U = _matriz_1q(nombre, theta)
            eje = nq - 1 - qs[0]                       # ejes del tensor: (q_{nq-1}, ..., q0)
            t = np.tensordot(U, psi.reshape([2] * nq), axes=([1], [eje]))
            psi = np.moveaxis(t, 0, eje).reshape(dim)
    return psi, perm_total


def entropia_entrelazamiento(psi: np.ndarray, k: int, nq: int = NUM_QUBITS) -> float:
    """Entropía de von Neumann (bits) del qubit k frente al resto (estado puro global)."""
    t = np.moveaxis(psi.reshape([2] * nq), nq - 1 - k, 0).reshape(2, -1)
    ev = np.linalg.eigvalsh(t @ t.conj().T)
    ev = ev[ev > 1e-15]
    return float(-np.sum(ev * np.log2(ev)))


# --- Ejecución -----------------------------------------------------------------

DESCRIPCION = describir(CIRCUITO)
statevector, perm_total = simular_numpy(CIRCUITO)
probabilidades = np.abs(statevector) ** 2

try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    from qiskit_aer import AerSimulator
    QISKIT_AVAILABLE = True
except ImportError:
    QISKIT_AVAILABLE = False
    print("[INFO] Qiskit no disponible - se usa el simulador numpy (mismo circuito)")

print("\n" + "=" * 80)
print("SIMULACIÓN DE CIRCUITO CUÁNTICO (3-QUBIT)")
print("=" * 80)
print(f"\nCircuito: {DESCRIPCION}")

if QISKIT_AVAILABLE:
    qc_base = QuantumCircuit(NUM_QUBITS)
    for nombre, qs, theta in CIRCUITO:
        if theta is None:
            getattr(qc_base, nombre)(*qs)
        else:
            getattr(qc_base, nombre)(theta, *qs)

    qc_demo = qc_base.copy()
    qc_demo.measure_all()          # sin registro clásico previo → bitstrings de 3 bits

    print("\nCircuito Cuántico Final:")
    print(qc_demo)
    print(f"\nPropiedades:")
    print(f"  Depth: {qc_demo.depth()}")
    print(f"  Gate Size: {qc_demo.size()}")

    sv_qiskit = np.asarray(Statevector(qc_base).data)
    fidelidad = abs(np.vdot(sv_qiskit, statevector)) ** 2
    print(f"\n  Verificación numpy vs Qiskit: fidelidad = {fidelidad:.12f}")
    assert fidelidad > 1 - 1e-10, "El simulador numpy y Qiskit no coinciden"

    counts = AerSimulator(seed_simulator=SEED).run(qc_demo, shots=SHOTS).result().get_counts()
else:
    muestras = rng.multinomial(SHOTS, probabilidades / probabilidades.sum())
    counts = {ket(i): int(c) for i, c in enumerate(muestras) if c > 0}

print("\n" + "=" * 80)
print("ESTADO VECTORIAL (antes de medición)  — kets |q2 q1 q0⟩")
print("=" * 80)
print(f"\nDimensión del estado vector: {len(statevector)}")
print("\nEstado vector (componentes con amplitud > 1e-4):")
for i, amplitude in enumerate(statevector):
    if abs(amplitude) > 1e-4:
        print(f"  |{ket(i)}⟩: {amplitude:.6f} (magnitud: {abs(amplitude):.6f}, "
              f"fase: {np.angle(amplitude):.4f} rad)")

print("\n" + "=" * 80)
print("PROBABILIDADES DE MEDICIÓN")
print("=" * 80)
print(f"\nEstados posibles con probabilidad > 1e-4:")
for i, prob in enumerate(probabilidades):
    if prob > 1e-4:
        print(f"  |{ket(i)}⟩: {prob:.6f} ({prob*100:.4f}%)")
print(f"\nSuma total de probabilidades: {np.sum(probabilidades):.10f} (debe ser 1.0)")

print("\n" + "=" * 80)
print(f"SIMULACIÓN CON MEDICIONES ({SHOTS} shots)")
print("=" * 80)
print(f"\nResultados de {SHOTS} ejecuciones:")
for bitstring, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {bitstring}: {count:4d} mediciones ({count / SHOTS * 100:.2f}%)")


# ============================================================================
# PARTICIÓN ALTO/BAJO DERIVADA DE LOS DATOS (antes: fija {0,2,4,7})
# ============================================================================

def particion_alto_bajo(probs: np.ndarray):
    """Mitad superior por probabilidad = 'altos'; el resto = 'bajos'."""
    orden = np.argsort(-probs, kind="stable")
    mitad = len(probs) // 2
    return sorted(orden[:mitad].tolist()), sorted(orden[mitad:].tolist())


estados_altos_idx, estados_bajos_idx = particion_alto_bajo(probabilidades)
amplitudes_altas = float(np.mean(np.abs(statevector[estados_altos_idx])))
amplitudes_bajas = float(np.mean(np.abs(statevector[estados_bajos_idx])))

# ============================================================================
# PARTE 1: ESTADO CUÁNTICO SIMULADO
# ============================================================================

print("\n" + "=" * 80)
print("ANÁLISIS DE ASIMETRÍA CUÁNTICA → COVARIANZA")
print("=" * 80)

ratio_asimetria = amplitudes_altas / amplitudes_bajas
prob_alta = float(np.sum(probabilidades[estados_altos_idx]))   # probabilidad TOTAL del grupo
prob_baja = float(np.sum(probabilidades[estados_bajos_idx]))

print(f"\nEstados altos: {[ket(i) for i in estados_altos_idx]}")
print(f"Estados bajos: {[ket(i) for i in estados_bajos_idx]}")
print(f"\nAmplitud alta (media de 4 estados): {amplitudes_altas:.6f}")
print(f"Amplitud baja (media de 4 estados): {amplitudes_bajas:.6f}")
print(f"Ratio de asimetría: {ratio_asimetria:.6f}")
print(f"\nProbabilidad alta (grupo): {prob_alta:.6f} ({prob_alta*100:.2f}%)")
print(f"Probabilidad baja (grupo): {prob_baja:.6f} ({prob_baja*100:.2f}%)")

# ============================================================================
# PARTE 2: FOCK SPACE (Definiciones)
# ============================================================================

class OccupationMode(Enum):
    """Enumeration of occupation number modes in Fock space."""
    BOSONIC = "bosonic"
    FERMIONIC = "fermionic"   # nota: aún no se aplica (los operadores son bosónicos)


@dataclass
class FockConfig:
    """Configuration for Fock space instantiation."""
    n_modes: int = 3
    n_max: int = 3
    mode_type: OccupationMode = OccupationMode.BOSONIC
    use_gray_code: bool = True


class FockBasis:
    """
    Fock space representation with second quantization operators.
    dim(H_Fock) = (n_max + 1)^n_modes
    """

    def __init__(self, config: FockConfig = None):
        if config is None:
            config = FockConfig()
        self.config = config
        self.n_modes = config.n_modes
        self.n_max = config.n_max
        self.dim = (config.n_max + 1) ** config.n_modes
        self.n_bits = max(1, (self.dim - 1).bit_length())   # ancho binario real de la base
        self._build_basis()
        self._precompute_operators()

    def _build_basis(self):
        basis_states = []
        for occupation in np.ndindex(*[self.n_max + 1] * self.n_modes):
            basis_states.append(occupation)
        self.basis_states = np.array(basis_states)
        assert len(self.basis_states) == self.dim
        self.state_to_index = {tuple(state): i for i, state in enumerate(self.basis_states)}
        self.index_to_state = {i: tuple(state) for i, state in enumerate(self.basis_states)}

    def _precompute_operators(self):
        self.creation_ops = {}
        self.annihilation_ops = {}
        for mode in range(self.n_modes):
            self.creation_ops[mode] = self._build_creation_op(mode)
            self.annihilation_ops[mode] = self._build_annihilation_op(mode)

    def _build_creation_op(self, mode: int) -> np.ndarray:
        op = np.zeros((self.dim, self.dim), dtype=complex)
        for i, state in enumerate(self.basis_states):
            state_list = list(state)
            if state_list[mode] < self.n_max:
                new_occupation = state_list[mode] + 1
                amplitude = np.sqrt(new_occupation)
                state_list[mode] = new_occupation
                j = self.state_to_index[tuple(state_list)]
                op[j, i] = amplitude
        return op

    def _build_annihilation_op(self, mode: int) -> np.ndarray:
        op = np.zeros((self.dim, self.dim), dtype=complex)
        for i, state in enumerate(self.basis_states):
            state_list = list(state)
            if state_list[mode] > 0:
                old_occupation = state_list[mode]
                amplitude = np.sqrt(old_occupation)
                state_list[mode] = old_occupation - 1
                j = self.state_to_index[tuple(state_list)]
                op[j, i] = amplitude
        return op

    def get_creation_op(self, mode: int) -> np.ndarray:
        if mode not in self.creation_ops:
            raise ValueError(f"Mode {mode} out of range [0, {self.n_modes-1}]")
        return self.creation_ops[mode].copy()

    def get_annihilation_op(self, mode: int) -> np.ndarray:
        if mode not in self.annihilation_ops:
            raise ValueError(f"Mode {mode} out of range [0, {self.n_modes-1}]")
        return self.annihilation_ops[mode].copy()

    def number_operator(self, mode: int) -> np.ndarray:
        a_dag = self.get_creation_op(mode)
        a = self.get_annihilation_op(mode)
        return a_dag @ a

    def total_number_operator(self) -> np.ndarray:
        N_total = np.zeros((self.dim, self.dim), dtype=complex)
        for mode in range(self.n_modes):
            N_total += self.number_operator(mode)
        return N_total

    def state_vector(self, occupation: Tuple[int, ...]) -> np.ndarray:
        if len(occupation) != self.n_modes:
            raise ValueError(f"Occupation must have {self.n_modes} elements")
        if any(n_occ > self.n_max for n_occ in occupation):
            raise ValueError(f"Occupation exceeds n_max = {self.n_max}")
        vec = np.zeros(self.dim, dtype=complex)
        idx = self.state_to_index[tuple(occupation)]
        vec[idx] = 1.0
        return vec

    def to_gray_code(self, state_index: int) -> int:
        return state_index ^ (state_index >> 1)

    def from_gray_code(self, gray: int) -> int:
        mask = gray
        while mask:
            mask >>= 1
            gray ^= mask
        return gray

    def occupation_to_binary(self, occupation: Tuple[int, ...]) -> str:
        state_index = self.state_to_index[tuple(occupation)]
        if self.config.use_gray_code:
            binary_rep = self.to_gray_code(state_index)
        else:
            binary_rep = state_index
        return format(binary_rep, f'0{self.n_bits}b')   # antes: n_modes*2 (solo válido con n_max=3)

    def get_basis_info(self) -> Dict:
        return {
            'dimension': self.dim,
            'n_modes': self.n_modes,
            'n_max_per_mode': self.n_max,
            'total_occupation_max': self.n_max * self.n_modes,
            'basis_states': self.basis_states.tolist(),
            'use_gray_code': self.config.use_gray_code,
            'mode_type': self.config.mode_type.value
        }


class FockStateVector:
    """Quantum state in Fock basis with helper methods."""

    def __init__(self, fock_basis: FockBasis, vector: np.ndarray = None):
        self.fock = fock_basis
        if vector is None:
            self.vec = np.zeros(fock_basis.dim, dtype=complex)
        else:
            if len(vector) != fock_basis.dim:
                raise ValueError(f"Vector dimension {len(vector)} != {fock_basis.dim}")
            self.vec = vector.astype(complex)

    def normalize(self):
        norm = np.linalg.norm(self.vec)
        if norm > 1e-10:
            self.vec /= norm
        return self

    def occupation_probabilities(self) -> Dict[Tuple[int, ...], float]:
        probs = {}
        for i, state in enumerate(self.fock.basis_states):
            prob = abs(self.vec[i])**2
            if prob > 1e-10:
                probs[tuple(state)] = prob
        return probs

    def expectation_value(self, operator: np.ndarray) -> complex:
        return np.vdot(self.vec, operator @ self.vec)   # ⟨ψ|O|ψ⟩ (antes faltaba conjugar)

    def fidelity(self, other: 'FockStateVector') -> float:
        overlap = np.abs(np.vdot(self.vec, other.vec))**2
        return np.clip(overlap, 0, 1)


def fock_ground_state(fock_basis: FockBasis) -> FockStateVector:
    """Return ground state |0, 0, ..., 0⟩."""
    occupation = tuple(0 for _ in range(fock_basis.n_modes))
    vec = fock_basis.state_vector(occupation)
    return FockStateVector(fock_basis, vec)


def fock_single_photon(fock_basis: FockBasis, mode: int) -> FockStateVector:
    """Return single photon in given mode |0, ..., 1_m, ..., 0⟩."""
    occupation = [0] * fock_basis.n_modes
    occupation[mode] = 1
    vec = fock_basis.state_vector(tuple(occupation))
    return FockStateVector(fock_basis, vec)


# ============================================================================
# FUNCIONES COMPARTIDAS (módulo y clase usan exactamente las mismas)
# ============================================================================

def matriz_estados(nq: int) -> np.ndarray:
    """Fila i = [q0, q1, ..., q_{nq-1}] del estado de base i (columna k = qubit k)."""
    return np.array([[bit(i, k) for k in range(nq)] for i in range(2 ** nq)], dtype=float)


def covarianza_ponderada(X: np.ndarray, w: np.ndarray) -> np.ndarray:
    """Cov_w[q_i, q_j] con pesos normalizados (distribución de probabilidad)."""
    w = w / w.sum()
    m = X.T @ w
    return (X * w[:, None]).T @ X - np.outer(m, m)


def factor_v1(probs: np.ndarray, altos, bajos) -> float:
    """(P_alto − P_bajo)/(P_alto + P_bajo) con probabilidades reales de cada grupo."""
    pa, pb = probs[altos].sum(), probs[bajos].sum()
    return float((pa - pb) / (pa + pb)) if (pa + pb) > 0 else 0.0


def s_teorico(i_n: float, o_n: float, ion: float) -> float:
    """⟨Z_q0⟩ tras H·RZ(ION)·RY(I_n)·RX(O_n): componente z de Bloch."""
    return float(np.sin(ion) * np.sin(o_n) - np.cos(ion) * np.sin(i_n) * np.cos(o_n))


def sigma_desde_s(s: float) -> np.ndarray:
    """Covarianza exacta de este circuito en función de la asimetría s."""
    return np.array([
        [0.25 - s**2 / 16, s / 8, s / 8],
        [s / 8,            0.25,  0.0  ],
        [s / 8,            0.0,   0.25 ],
    ])


# ============================================================================
# PARTE 3: DECODIFICACIÓN DIRECTA (Single-Pass)
# ============================================================================

print("\n" + "=" * 80)
print("DECODIFICACIÓN DIRECTA (Single-Pass Tensor Contraction)")
print("=" * 80)

estado_matrix = matriz_estados(NUM_QUBITS)
etiquetas_amplitud = probabilidades          # probabilidad real de cada estado

print("\nEstados y sus probabilidades:")
for i in range(DIM):
    tipo = "ALTO" if i in estados_altos_idx else "BAJO"
    print(f"  |{ket(i)}⟩ ({i}): {etiquetas_amplitud[i]:.6f} "
          f"({etiquetas_amplitud[i]*100:.2f}%) [{tipo}]")

# ============================================================================
# PASO 2: Característica diferenciadora (derivada del circuito, no supuesta)
# ============================================================================

print("\n" + "=" * 80)
print("EXTRACCIÓN DE CARACTERÍSTICAS (Feature Differentiation)")
print("=" * 80)

# q0 ANTES de CCX/CSWAP/CCX: es el único qubit sesgado por las rotaciones,
# así que es lo que separa los dos grupos.
perm_inversa = np.argsort(perm_total)
caracteristica = np.array([bit(int(perm_inversa[i]), 0) for i in range(DIM)])
forma_booleana = np.array([bit(i, 0) ^ (bit(i, 1) & bit(i, 2)) for i in range(DIM)])
assert np.array_equal(caracteristica, forma_booleana)

print(f"\nCaracterística diferenciadora: q0 antes de las compuertas reversibles")
print(f"  = q_0 ⊕ (q_1·q_2) en los qubits finales")
for i in range(DIM):
    print(f"  |{ket(i)}⟩: q_0⊕(q_1·q_2) = {caracteristica[i]}, P(|i⟩) = {etiquetas_amplitud[i]:.6f}")

s_medido = float(probabilidades[caracteristica == 0].sum() - probabilidades[caracteristica == 1].sum())

# ============================================================================
# PASO 3: COVARIANZA qubit × probabilidad
# ============================================================================

print("\n" + "=" * 80)
print("COVARIANZA (Qubits × Probabilidad)")
print("=" * 80)

# Covarianza sobre los 8 estados de base (medida uniforme). Equivale a (⟨q_k⟩ − 1/2)/8.
cov_qubits_amplitud = np.array([
    np.mean(estado_matrix[:, k] * etiquetas_amplitud)
    - np.mean(estado_matrix[:, k]) * np.mean(etiquetas_amplitud)
    for k in range(NUM_QUBITS)
])

for k in range(NUM_QUBITS):
    print(f"Covarianza(q_{k}, P) = {cov_qubits_amplitud[k]:+.6f}")

k_max = int(np.argmax(np.abs(cov_qubits_amplitud)))
print(f"\nInterpretación:\n  → q_{k_max} es el qubit que explica la asimetría")

# ============================================================================
# PASO 4: MATRIZ DE COVARIANZA INTER-QUBIT (ponderada por probabilidad)
# ============================================================================

print("\n" + "=" * 80)
print("MATRIZ DE COVARIANZA INTER-QUBIT (3×3 ponderada)  — filas/columnas q0,q1,q2")
print("=" * 80)

cov_matrix = covarianza_ponderada(estado_matrix, etiquetas_amplitud)
print("\nMatriz de Covarianza Ponderada (Cov[q_i, q_j]):")
print(cov_matrix)

eigenvalores, eigenvectores = eigh(cov_matrix)
print(f"\nEigenvalores: {eigenvalores}")
print(f"Eigenvectores:\n{eigenvectores}")

# ============================================================================
# PASO 5: COMPRESIÓN A ESCALAR ÚNICO
# ============================================================================

print("\n" + "=" * 80)
print("COMPRESIÓN A ESCALAR ÚNICO (Single Number)")
print("=" * 80)

factor_covarianza_v1 = factor_v1(probabilidades, estados_altos_idx, estados_bajos_idx)
factor_covarianza_v2 = float(np.trace(cov_matrix))
factor_covarianza_v3 = float(np.max(eigenvalores))
factor_covarianza_v4 = float(np.linalg.norm(cov_matrix, 'fro'))

print(f"\nFactores de Covarianza derivados:")
print(f"  v1 (Ratio ponderado = |s|): {factor_covarianza_v1:.6f}")
print(f"  v2 (Traza):                 {factor_covarianza_v2:.6f}")
print(f"  v3 (Eigenvalor principal):  {factor_covarianza_v3:.6f}")
print(f"  v4 (Norma Frobenius):       {factor_covarianza_v4:.6f}")

factor_normalized = factor_covarianza_v1 / phi
print(f"\nFactor normalizado por φ = {factor_normalized:.6f}")

# ============================================================================
# PASO 6: VERIFICACIÓN Y RELACIÓN CON H7
# ============================================================================

print("\n" + "=" * 80)
print("VERIFICACIÓN: RELACIÓN CON H7 FRAMEWORK")
print("=" * 80)

s_h7 = s_teorico(i_n, o_n, ion)
print(f"\nAsimetría s (medida, P[g=0]−P[g=1]): {s_medido:+.10f}")
print(f"Asimetría s (fórmula H7):            {s_h7:+.10f}")
print(f"  s = sin(ION)·sin(O_n) − cos(ION)·sin(I_n)·cos(O_n)")
print(f"  |diferencia| = {abs(s_medido - s_h7):.2e}")

print(f"\nComparativa con H7 Constants:")
print(f"  Ratio de asimetría: {ratio_asimetria:.6f}   (= φ solo si |s| = 1/√5 = {1/np.sqrt(5):.6f})")
print(f"  cos(φ):             {np.cos(phi):.6f}")
print(f"  φ:                  {phi:.6f}")
print(f"  O_n_integrity:      {O_n_integrity:.6f}")

# Antes se llamaba "entrelazamiento"; |a_alta − a_baja| es solo un contraste de
# magnitudes (≠ 0 incluso en estados producto). La medida real está abajo.
contraste_amplitud = abs(amplitudes_altas - amplitudes_bajas)
entropias = [entropia_entrelazamiento(statevector, k) for k in range(NUM_QUBITS)]
print(f"\nContraste de amplitud |a_alta − a_baja|: {contraste_amplitud:.6f}")
for k, S in enumerate(entropias):
    print(f"Entropía de entrelazamiento S(q_{k} | resto): {S:.6f} bits")

# ============================================================================
# PASO 7: MATRIZ DE COVARIANZA RECONSTRUIDA DESDE s (Validación)
# ============================================================================

print("\n" + "=" * 80)
print("MATRIZ DE COVARIANZA RECONSTRUIDA DESDE s (Validation)")
print("=" * 80)

cov_reconstructed = sigma_desde_s(s_h7)
print("\nΣ(s) reconstruida solo con la asimetría s:")
print(cov_reconstructed)
diff = np.linalg.norm(cov_matrix - cov_reconstructed, 'fro')
print(f"\nDiferencia con matriz empírica (Frobenius norm): {diff:.2e}")


# ============================================================================
# QUANTUM COVARIANCE DECODER - Clase Principal
# ============================================================================

class QuantumCovarianceDecoder:
    """
    Decodificador de covarianza desde asimetría cuántica.
    Recibe el vector de probabilidades real (no dos escalares) y deriva todo de él.
    """

    def __init__(self, probabilidades, num_qubits=3, description="", statevector=None):
        self.probabilidades = np.asarray(probabilidades, dtype=float)
        self.num_qubits = num_qubits
        self.description = description

        self.estados_altos, self.estados_bajos = particion_alto_bajo(self.probabilidades)
        mags = np.abs(statevector) if statevector is not None else np.sqrt(self.probabilidades)
        self.amplitudes_altas = float(np.mean(mags[self.estados_altos]))
        self.amplitudes_bajas = float(np.mean(mags[self.estados_bajos]))

        self.ratio_asimetria = self.amplitudes_altas / self.amplitudes_bajas
        self.prob_alta = float(self.probabilidades[self.estados_altos].sum())
        self.prob_baja = float(self.probabilidades[self.estados_bajos].sum())
        self.contraste_amplitud = abs(self.amplitudes_altas - self.amplitudes_bajas)

        self.estado_matrix = None
        self.cov_matrix = None
        self.eigenvalores = None
        self.eigenvectores = None

    def deducir(self):
        """Realizar deducción completa en un solo movimiento"""
        self.estado_matrix = matriz_estados(self.num_qubits)
        self.cov_matrix = covarianza_ponderada(self.estado_matrix, self.probabilidades)
        self.eigenvalores, self.eigenvectores = eigh(self.cov_matrix)
        return {
            "ratio_asimetria": self.ratio_asimetria,
            "covarianza": self.cov_matrix,
            "eigenvalores": self.eigenvalores,
            "eigenvectores": self.eigenvectores,
        }

    def factor_covarianza(self, version=1):
        """1. Ratio ponderado  2. Traza  3. Eigenvalor principal  4. Norma Frobenius"""
        if self.cov_matrix is None:
            self.deducir()
        if version == 1:
            return factor_v1(self.probabilidades, self.estados_altos, self.estados_bajos)
        elif version == 2:
            return float(np.trace(self.cov_matrix))
        elif version == 3:
            return float(np.max(self.eigenvalores))
        elif version == 4:
            return float(np.linalg.norm(self.cov_matrix, 'fro'))
        raise ValueError(f"Version {version} not supported")

    def normalizar_por_phi(self):
        """Normalizar todos los factores por φ"""
        return [self.factor_covarianza(v) / phi for v in [1, 2, 3, 4]]

    def h7_signature(self):
        """Generar firma H7 a partir de la covarianza deducida"""
        if self.cov_matrix is None:
            self.deducir()
        eig_principal = float(np.max(self.eigenvalores))
        traza = float(np.trace(self.cov_matrix))
        determinante = float(np.linalg.det(self.cov_matrix))
        sig_h7 = (eig_principal + O_n_integrity) * phi / (1 + DRIFT_072 / 10)
        return {
            "eigenvalor_principal": eig_principal,
            "traza": traza,
            "determinante": determinante,
            "h7_signature": float(sig_h7),
            "normalized_by_phi": float(sig_h7 / phi),
        }

    def reporte_completo(self):
        """Generar reporte completo de la deducción"""
        if self.cov_matrix is None:
            self.deducir()

        print("\n" + "=" * 80)
        print("QUANTUM COVARIANCE DECODER - H7 FRAMEWORK")
        print("=" * 80)
        print(f"Descripción: {self.description}")
        print(f"Qubits: {self.num_qubits}")

        print("\n" + "-" * 80)
        print("ASIMETRÍA CUÁNTICA")
        print("-" * 80)
        print(f"Amplitud alta:  {self.amplitudes_altas:.6f} → P(grupo) = {self.prob_alta:.6f}")
        print(f"Amplitud baja:  {self.amplitudes_bajas:.6f} → P(grupo) = {self.prob_baja:.6f}")
        print(f"Ratio:          {self.ratio_asimetria:.6f}")
        print(f"Contraste:      {self.contraste_amplitud:.6f}")

        print("\n" + "-" * 80)
        print(f"MATRIZ DE COVARIANZA DEDUCIDA ({self.num_qubits}×{self.num_qubits})")
        print("-" * 80)
        print(self.cov_matrix)

        print("\n" + "-" * 80)
        print("DESCOMPOSICIÓN ESPECTRAL")
        print("-" * 80)
        print(f"Eigenvalores: {self.eigenvalores}")
        print(f"Eigenvector principal (máximo eigenvalor):")
        print(f"  {self.eigenvectores[:, np.argmax(self.eigenvalores)]}")

        print("\n" + "-" * 80)
        print("FACTORES DE COVARIANZA COMPRIMIDA")
        print("-" * 80)
        for v in [1, 2, 3, 4]:
            factor = self.factor_covarianza(v)
            print(f"  v{v}: {factor:.6f} → normalizado: {factor / phi:.6f}")

        print("\n" + "-" * 80)
        print("FIRMA H7")
        print("-" * 80)
        for key, val in self.h7_signature().items():
            print(f"  {key}: {val:.6f}")
        print("\n" + "=" * 80)


# ============================================================================
# APLICACIÓN: Deducir covarianza del circuito anterior
# ============================================================================

print("\n" + "=" * 80)
print("INSTANCIA DECODER: ANÁLISIS COMPLETO")
print("=" * 80)

decoder = QuantumCovarianceDecoder(
    probabilidades=probabilidades,
    num_qubits=NUM_QUBITS,
    description=DESCRIPCION,
    statevector=statevector,
)
resultado = decoder.deducir()
decoder.reporte_completo()

# Módulo y clase deben dar lo mismo (antes eran dos cálculos distintos)
assert np.allclose(decoder.cov_matrix, cov_matrix)
assert np.isclose(decoder.factor_covarianza(1), factor_covarianza_v1)

# ================================================================
# Exportar a JSON
# ================================================================

export_dict = {
    "n": n,
    "seed": SEED,
    "circuito": decoder.description,
    "parametros_h7": {"I_n": float(i_n), "O_n": float(o_n), "ION": float(ion),
                      "p": float(p), "q": float(q), "O_n_integrity": float(O_n_integrity)},
    "asimetria": {
        "s": s_h7,
        "amplitud_alta": decoder.amplitudes_altas,
        "amplitud_baja": decoder.amplitudes_bajas,
        "ratio": float(decoder.ratio_asimetria),
        "contraste_amplitud": float(decoder.contraste_amplitud),
        "entropia_entrelazamiento_bits": entropias,
    },
    "covarianza": {
        "matriz": decoder.cov_matrix.tolist(),
        "eigenvalores": decoder.eigenvalores.tolist(),
        "traza": float(np.trace(decoder.cov_matrix)),
        "determinante": float(np.linalg.det(decoder.cov_matrix)),
        "error_vs_sigma_s": float(diff),
    },
    "h7_signature": decoder.h7_signature(),
}

output_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
output_path = os.path.join(output_dir, "h7_covarianza.json")
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(export_dict, f, indent=2, ensure_ascii=False)
print(f"\n✓ Resultado exportado a: {output_path}")

# ============================================================================
# RESUMEN FINAL
# ============================================================================

print("\n" + "=" * 80)
print("RESUMEN: COVARIANZA EN UN SOLO MOVIMIENTO")
print("=" * 80)

summary = {
    "n": n,
    "Asimetría s (fórmula H7)": s_h7,
    "Ratio de amplitudes": decoder.ratio_asimetria,
    "Factor de Covarianza (v1 = |s|)": decoder.factor_covarianza(1),
    "Factor Normalizado (φ)": decoder.factor_covarianza(1) / phi,
    "Contraste de amplitud": decoder.contraste_amplitud,
    "S(q_0 | resto) [bits]": entropias[0],
    "Eigenvalor Principal": decoder.factor_covarianza(3),
    "Traza Covarianza": decoder.factor_covarianza(2),
    "Error Σ(s) vs empírica": diff,
    "Característica Diferenciadora": "q_0 ⊕ (q_1·q_2)  (= q_0 antes de CCX/CSWAP/CCX)",
}

for key, val in summary.items():
    if isinstance(val, float):
        print(f"{key:.<40} {val:.6f}" if abs(val) > 1e-4 or val == 0 else f"{key:.<40} {val:.2e}")
    else:
        print(f"{key:.<40} {val}")

print("\n" + "=" * 80)
print("✓ CONCLUSIÓN: Single-pass deduction completada exitosamente")
print("=" * 80)

[INIT] n = 945  (SEED = 7, N_FIJO = None)
[INIT] Golden Ratio φ = 1.618034
[INIT] I_n = -1.000000, O_n = -0.991258, ION = 0.991258, p = 0.000000, q = 1.982516
[INIT] O_n_integrity = -0.991258
[INIT] DRIFT_072 = 0.716815

SIMULACIÓN DE CIRCUITO CUÁNTICO (3-QUBIT)

Circuito: H⊗³ → RZ(+0.9913)·RY(-1.0000)·RX(-0.9913) en q0 → CCX(q2,q0→q1) → CSWAP(q1: q2↔q0) → CCX(q0,q1→q2)

Circuito Cuántico Final:
        ┌───┐┌─────────────┐┌────────┐┌──────────────┐              ░ ┌─┐      
   q_0: ┤ H ├┤ Rz(0.99126) ├┤ Ry(-1) ├┤ Rx(-0.99126) ├──■───X───■───░─┤M├──────
        ├───┤└─────────────┘└────────┘└──────────────┘┌─┴─┐ │   │   ░ └╥┘┌─┐   
   q_1: ┤ H ├─────────────────────────────────────────┤ X ├─■───■───░──╫─┤M├───
        ├───┤                                         └─┬─┘ │ ┌─┴─┐ ░  ║ └╥┘┌─┐
   q_2: ┤ H ├───────────────────────────────────────────■───X─┤ X ├─░──╫──╫─┤M├
        └───┘                                                 └───┘ ░  ║  ║ └╥┘
meas: 3/═════════════════════════════════

In [4]:
"""
H7 UNIFIED FRAMEWORK - COVARIANZA + METRIPLEX ORACLE
=====================================================
Jako's complete H7 Framework combining:
1. Quantum Covariance Deduction (from asymmetry)
2. Metriplex Momentum Oracle (Simon's algorithm)
3. MOD 6 Topology Encoding
4. H7 Conservation Laws

VERSIÓN INTEGRADA Y SINCRONIZADA
"""

import numpy as np
import math
import json
import struct
from typing import Tuple, Dict, List, Callable, Optional
from dataclasses import dataclass
from enum import Enum
from scipy.linalg import eigh
import warnings

# ============================================================================
# GLOBAL CONSTANTS
# ============================================================================

phi = (1 + math.sqrt(5)) / 2
H_MAX = 7
O_n_integrity = 0.3624
DRIFT_072 = 7 - 2 * math.pi

print(f"\n{'='*80}")
print(f"H7 UNIFIED FRAMEWORK - Initialization")
print(f"{'='*80}")
print(f"[CONST] φ (Golden Ratio) = {phi:.6f}")
print(f"[CONST] H_MAX = {H_MAX}")
print(f"[CONST] O_n_integrity = {O_n_integrity:.6f}")
print(f"[CONST] DRIFT_072 = {DRIFT_072:.6f}")

# ============================================================================
# PART 1: QUANTUM COVARIANCE DECODER
# ============================================================================

class QuantumCovarianceDecoder:
    """
    Decodificador de covarianza desde asimetría cuántica.
    Single-pass tensor contraction approach.
    """
    
    def __init__(self, amplitudes_altas, amplitudes_bajas, num_qubits=3, description=""):
        self.amplitudes_altas = amplitudes_altas
        self.amplitudes_bajas = amplitudes_bajas
        self.num_qubits = num_qubits
        self.description = description
        
        # Derivadas
        self.ratio_asimetria = amplitudes_altas / amplitudes_bajas
        self.prob_alta = amplitudes_altas ** 2
        self.prob_baja = amplitudes_bajas ** 2
        self.medida_entrelazamiento = abs(amplitudes_altas - amplitudes_bajas)
        
        # Matrices
        self.estado_matrix = None
        self.etiquetas_amplitud = None
        self.cov_matrix = None
        self.eigenvalores = None
        self.eigenvectores = None
        
    def _construir_matriz_estados(self):
        num_estados = 2 ** self.num_qubits
        self.estado_matrix = np.array([
            [int(b) for b in format(i, f'0{self.num_qubits}b')]
            for i in range(num_estados)
        ], dtype=float)
        
    def _etiquetar_amplitudes(self):
        num_estados = 2 ** self.num_qubits
        if self.num_qubits == 3:
            estados_altos = {0, 2, 4, 7}
        else:
            estados_altos = {i for i in range(num_estados) if bin(i).count('1') % 2 == 0}
        
        self.etiquetas_amplitud = np.array([
            self.prob_alta if i in estados_altos else self.prob_baja
            for i in range(num_estados)
        ])
        
    def _calcular_covarianza_ponderada(self):
        self.cov_matrix = np.zeros((self.num_qubits, self.num_qubits))
        
        for i in range(self.num_qubits):
            for j in range(self.num_qubits):
                X_i = self.estado_matrix[:, i]
                X_j = self.estado_matrix[:, j]
                
                E_XiXj = np.sum(X_i * X_j * self.etiquetas_amplitud)
                E_Xi = np.sum(X_i * self.etiquetas_amplitud)
                E_Xj = np.sum(X_j * self.etiquetas_amplitud)
                
                self.cov_matrix[i, j] = E_XiXj - E_Xi * E_Xj
                
    def _eigendescomposicion(self):
        self.eigenvalores, self.eigenvectores = eigh(self.cov_matrix)
        
    def deducir(self):
        self._construir_matriz_estados()
        self._etiquetar_amplitudes()
        self._calcular_covarianza_ponderada()
        self._eigendescomposicion()
        
        return {
            "ratio_asimetria": self.ratio_asimetria,
            "covarianza": self.cov_matrix,
            "eigenvalores": self.eigenvalores,
            "eigenvectores": self.eigenvectores,
        }
        
    def factor_covarianza(self, version=1):
        if self.cov_matrix is None:
            self.deducir()
            
        if version == 1:
            p_alto = self.prob_alta * (2 ** self.num_qubits - 1)
            p_bajo = self.prob_baja * (2 ** self.num_qubits - 1)
            return (p_alto - p_bajo) / (p_alto + p_bajo) if (p_alto + p_bajo) > 0 else 0
        elif version == 2:
            return np.trace(self.cov_matrix)
        elif version == 3:
            return np.max(self.eigenvalores) if self.eigenvalores is not None else 0
        elif version == 4:
            return np.linalg.norm(self.cov_matrix, 'fro')
        else:
            raise ValueError(f"Version {version} not supported")
            
    def h7_signature(self):
        if self.cov_matrix is None:
            self.deducir()
        
        eig_principal = np.max(self.eigenvalores)
        traza = np.trace(self.cov_matrix)
        determinante = np.linalg.det(self.cov_matrix)
        
        sig_h7 = (eig_principal + O_n_integrity) * phi / (1 + DRIFT_072 / 10)
        
        return {
            "eigenvalor_principal": float(eig_principal),
            "traza": float(traza),
            "determinante": float(determinante),
            "h7_signature": float(sig_h7),
            "normalized_by_phi": float(sig_h7 / phi),
        }


# ============================================================================
# PART 2: ENERGY PROFILE & METRIPLEX ORACLE
# ============================================================================

class EnergyProfile(Enum):
    """Predefined energy normalization profiles."""
    LINEAR = "linear"
    QUADRATIC = "quadratic"
    METRIPLEX = "metriplex"
    CUSTOM = "custom"


@dataclass
class MetriplexConfig:
    """Configuration for metriplex oracle."""
    momentum_range: Tuple[int, int] = (1, 6)
    energy_profile: EnergyProfile = EnergyProfile.METRIPLEX
    normalization_target: float = 0.45
    collision_groups: Optional[Dict[str, List[int]]] = None

    def __post_init__(self):
        if self.collision_groups is None:
            self.collision_groups = {
                'A': [1, 2, 3],
                'B': [4, 5, 6]
            }


class MetriplexOracle:
    """
    Metriplex momentum oracle for second quantization.
    Maps momentum states to energy vectors with hidden 2-to-1 function.
    """

    def __init__(self, config: MetriplexConfig = None):
        if config is None:
            config = MetriplexConfig()
        self.config = config
        self.p_min, self.p_max = config.momentum_range
        self._build_energy_map()
        self._build_collision_map()

    def _build_energy_map(self):
        self.energy_map = {}
        for p in range(self.p_min, self.p_max + 1):
            if self.config.energy_profile == EnergyProfile.LINEAR:
                raw_energy = p / self.p_max
            elif self.config.energy_profile == EnergyProfile.QUADRATIC:
                raw_energy = (p / self.p_max) ** 2
            elif self.config.energy_profile == EnergyProfile.METRIPLEX:
                alpha = 1.2
                raw_energy = self.config.normalization_target * (p / self.p_max) ** alpha
            else:
                raise ValueError(f"Unknown energy profile: {self.config.energy_profile}")
            self.energy_map[p] = raw_energy

        mean_energy = np.mean([self.energy_map[p] for p in range(self.p_min, self.p_max + 1)])
        if abs(mean_energy - self.config.normalization_target) > 0.01:
            warnings.warn(
                f"Mean energy {mean_energy:.4f} deviates from target "
                f"{self.config.normalization_target:.4f}"
            )

    def _build_collision_map(self):
        self.collision_map = {}
        self.output_groups = {}
        for group_name, momenta in self.config.collision_groups.items():
            for p in momenta:
                self.collision_map[p] = group_name
            group_index = list(self.config.collision_groups.keys()).index(group_name)
            n_groups = len(self.config.collision_groups)
            output_vec = np.zeros(n_groups)
            output_vec[group_index] = 1.0
            self.output_groups[group_name] = output_vec

    def forward(self, momentum: int) -> Tuple[str, np.ndarray, float]:
        if momentum < self.p_min or momentum > self.p_max:
            raise ValueError(f"Momentum {momentum} out of range [{self.p_min}, {self.p_max}]")
        group = self.collision_map[momentum]
        output_vec = self.output_groups[group]
        energy = self.energy_map[momentum]
        return group, output_vec, energy

    def collide_pair(self, p1: int, p2: int) -> bool:
        return self.collision_map[p1] == self.collision_map[p2]

    def get_collision_partners(self, p: int) -> List[int]:
        group = self.collision_map[p]
        return self.config.collision_groups[group]

    def symmetry_string(self) -> int:
        group_lists = list(self.config.collision_groups.values())
        xor_accumulator = 0
        for group in group_lists:
            for i, p1 in enumerate(group):
                for p2 in group[i+1:]:
                    xor_accumulator |= (p1 ^ p2)
        return xor_accumulator

    def get_oracle_info(self) -> Dict:
        return {
            'momentum_range': self.config.momentum_range,
            'n_groups': len(self.config.collision_groups),
            'collision_groups': {k: v for k, v in self.config.collision_groups.items()},
            'symmetry_string': self.symmetry_string(),
            'energy_profile': self.config.energy_profile.value,
            'normalization_target': self.config.normalization_target,
            'energy_map': {p: self.energy_map[p] for p in range(self.p_min, self.p_max + 1)},
            'collision_structure': {p: self.collision_map[p] for p in range(self.p_min, self.p_max + 1)}
        }


# ============================================================================
# PART 3: H7 CONSERVATION LAW
# ============================================================================

class H7Conservation:
    """
    Mechanism I: H7 Entanglement Conservation
    States are paired by: x ↔ (7 ⊕ x) in 3-qubit Hilbert space.
    """

    CONSERVATION_CONSTANT = 7

    @staticmethod
    def partner_state(state: int) -> int:
        if not (0 <= state <= 7):
            raise ValueError("State must be in [0, 7]")
        return H7Conservation.CONSERVATION_CONSTANT ^ state

    @staticmethod
    def verify_pairing(state_a: int, state_b: int) -> bool:
        return state_b == H7Conservation.partner_state(state_a)

    @staticmethod
    def pairing_table() -> Dict[int, int]:
        return {i: H7Conservation.partner_state(i) for i in range(8)}


# ============================================================================
# PART 4: TOPOLOGY ENCODER
# ============================================================================

class BigEndianHexadecimalEncoder:
    """Encodes values to hexadecimal with big-endian byte ordering."""

    @staticmethod
    def to_hex_uint16(value: int) -> str:
        assert 0 <= value <= 65535, f"uint16 range: 0-65535, got {value}"
        return f"{value:04X}"

    @staticmethod
    def from_hex_uint16(hex_str: str) -> int:
        value = int(hex_str, 16)
        assert 0 <= value <= 65535
        return value


class TopologicalBigEndianEncoder:
    """Packs topology and converts to exact hexadecimal formats."""

    TERNARY_TO_BITS = {-1: 0b00, 0: 0b01, +1: 0b10}

    topology_entries = [
        {'index': 1, 'pair': 6, 'winding': 0, 'mapping': 0, 'ternary_weight': 1,  'discrete_phase_fragment': 0},
        {'index': 5, 'pair': 2, 'winding': 0, 'mapping': 0, 'ternary_weight': 1,  'discrete_phase_fragment': 1},
        {'index': 3, 'pair': 4, 'winding': 0, 'mapping': 0, 'ternary_weight': 1,  'discrete_phase_fragment': 6},
        {'index': 4, 'pair': 3, 'winding': 2, 'mapping': 1, 'ternary_weight': -1, 'discrete_phase_fragment': 5},
        {'index': 5, 'pair': 2, 'winding': 2, 'mapping': 1, 'ternary_weight': -1, 'discrete_phase_fragment': 2},
        {'index': 6, 'pair': 1, 'winding': 2, 'mapping': 1, 'ternary_weight': -1, 'discrete_phase_fragment': 3},
        {'index': 2, 'pair': 3, 'winding': 0, 'mapping': 0, 'ternary_weight': 0,  'discrete_phase_fragment': 4},
    ]

    @staticmethod
    def pack_topology(index: int, pair: int, winding: int, mapping: int, 
                     ternary_weight: int, discrete_phase_fragment: int) -> int:
        """Packs topology into 14 bits."""
        assert 1 <= index <= 6
        assert 1 <= pair <= 6
        assert winding in [0, 2]
        assert mapping in [0, 1]
        assert ternary_weight in [-1, 0, 1]
        assert 0 <= discrete_phase_fragment <= 7

        value = 0
        value |= (index - 1) << 0
        value |= (pair - 1) << 3
        value |= (winding // 2) << 6
        value |= mapping << 8
        value |= TopologicalBigEndianEncoder.TERNARY_TO_BITS[ternary_weight] << 9
        value |= discrete_phase_fragment << 11

        return value

    @staticmethod
    def unpack_topology(value: int) -> Dict:
        """Unpacks topological state"""
        index = ((value >> 0) & 0b111) + 1
        pair = ((value >> 3) & 0b111) + 1
        winding = ((value >> 6) & 0b1) * 2
        mapping = (value >> 8) & 0b1
        weight_bits = (value >> 9) & 0b11
        ternary_weight = {0b00: -1, 0b01: 0, 0b10: 1}.get(weight_bits, 0)
        discrete_phase_fragment = (value >> 11) & 0b111
        cycle_phase_derived = (discrete_phase_fragment / 3.5) - 1.0

        return {
            'index': index,
            'pair': pair,
            'winding': winding,
            'mapping': mapping,
            'ternary_weight': ternary_weight,
            'discrete_phase_fragment': discrete_phase_fragment,
            'cycle_phase_derived': cycle_phase_derived,
        }


# ============================================================================
# PART 5: UNIFIED INTEGRATION & DEMONSTRATION
# ============================================================================

class H7UnifiedSystem:
    """
    Unified system integrating covarianza decoder + metriplex oracle.
    """
    
    def __init__(self, amplitudes_altas: float, amplitudes_bajas: float):
        print(f"\n[UNIFIED] Initializing H7 Unified System")
        
        # Initialize covarianza decoder
        self.decoder = QuantumCovarianceDecoder(
            amplitudes_altas=amplitudes_altas,
            amplitudes_bajas=amplitudes_bajas,
            num_qubits=3,
            description="3-qubit circuit for H7 analysis"
        )
        
        # Initialize metriplex oracle
        self.oracle = MetriplexOracle()
        
        # Run deductions
        self.decoder.deducir()
        
    def generate_report(self) -> Dict:
        """Generate unified report."""
        return {
            "covarianza": {
                "ratio_asimetria": float(self.decoder.ratio_asimetria),
                "matriz": self.decoder.cov_matrix.tolist(),
                "eigenvalores": self.decoder.eigenvalores.tolist(),
                "h7_signature": self.decoder.h7_signature()
            },
            "metriplex_oracle": self.oracle.get_oracle_info(),
            "h7_conservation": H7Conservation.pairing_table(),
            "topology": {
                "entries": TopologicalBigEndianEncoder.topology_entries,
            }
        }


def main():
    """Main execution function."""
    print(f"\n{'#'*80}")
    print("# H7 UNIFIED FRAMEWORK - INTEGRATED EXECUTION")
    print(f"{'#'*80}")
    
    # Initialize system with synthetic data
    amplitudes_altas = 0.479653
    amplitudes_bajas = 0.141186
    
    system = H7UnifiedSystem(amplitudes_altas, amplitudes_bajas)
    
    # Print results
    print(f"\n{'-'*80}")
    print("COVARIANZA DEDUCTION")
    print(f"{'-'*80}")
    print(f"Ratio de asimetría: {system.decoder.ratio_asimetria:.6f}")
    print(f"Matriz de covarianza:")
    print(system.decoder.cov_matrix)
    print(f"Eigenvalores: {system.decoder.eigenvalores}")
    
    print(f"\n{'-'*80}")
    print("METRIPLEX ORACLE")
    print(f"{'-'*80}")
    info = system.oracle.get_oracle_info()
    print(f"Momentum range: {info['momentum_range']}")
    print(f"Collision groups: {info['collision_groups']}")
    print(f"Symmetry string: {info['symmetry_string']}")
    
    print(f"\n{'-'*80}")
    print("H7 CONSERVATION")
    print(f"{'-'*80}")
    pairing = H7Conservation.pairing_table()
    for state, partner in sorted(pairing.items()):
        print(f"  |{state}⟩ ↔ |{partner}⟩")
    
    print(f"\n{'-'*80}")
    print("TOPOLOGY ENCODING")
    print(f"{'-'*80}")
    for i, entry in enumerate(TopologicalBigEndianEncoder.topology_entries):
        value = TopologicalBigEndianEncoder.pack_topology(
            index=entry['index'],
            pair=entry['pair'],
            winding=entry['winding'],
            mapping=entry['mapping'],
            ternary_weight=entry['ternary_weight'],
            discrete_phase_fragment=entry['discrete_phase_fragment']
        )
        hex_val = BigEndianHexadecimalEncoder.to_hex_uint16(value & 0xFFFF)
        print(f"  n={i}: index={entry['index']}, pair={entry['pair']}, "
              f"winding={entry['winding']}, mapping={entry['mapping']}, "
              f"weight={entry['ternary_weight']:>2} → {hex_val}")
    
    # Generate and export report
    print(f"\n{'-'*80}")
    print("GENERATING UNIFIED REPORT")
    print(f"{'-'*80}")
    report = system.generate_report()
    
    output_file = "/kaggle/working/h7_unified_report.json"
    with open(output_file, 'w') as f:
        # Convert numpy arrays to lists for JSON serialization
        report_clean = {
            "covarianza": {
                "ratio_asimetria": report["covarianza"]["ratio_asimetria"],
                "matriz": report["covarianza"]["matriz"],
                "eigenvalores": report["covarianza"]["eigenvalores"],
                "h7_signature": report["covarianza"]["h7_signature"]
            },
            "metriplex_oracle": report["metriplex_oracle"],
            "h7_conservation": report["h7_conservation"],
        }
        json.dump(report_clean, f, indent=2)
    
    print(f"✓ Report exported to: {output_file}")
    
    print(f"\n{'#'*80}")
    print("✓ H7 UNIFIED FRAMEWORK EXECUTION COMPLETED")
    print(f"{'#'*80}")


if __name__ == '__main__':
    main()


H7 UNIFIED FRAMEWORK - Initialization
[CONST] φ (Golden Ratio) = 1.618034
[CONST] H_MAX = 7
[CONST] O_n_integrity = 0.362400
[CONST] DRIFT_072 = 0.716815

################################################################################
# H7 UNIFIED FRAMEWORK - INTEGRATED EXECUTION
################################################################################

[UNIFIED] Initializing H7 Unified System

--------------------------------------------------------------------------------
COVARIANZA DEDUCTION
--------------------------------------------------------------------------------
Ratio de asimetría: 3.397313
Matriz de covarianza:
[[ 2.50000000e-01 -4.87005949e-07  1.05066475e-01]
 [-4.87005949e-07  2.50000000e-01  1.05066475e-01]
 [ 1.05066475e-01  1.05066475e-01  2.05844316e-01]]
Eigenvalores: [0.07770424 0.25000049 0.37813958]

--------------------------------------------------------------------------------
METRIPLEX ORACLE
---------------------------------------------------------

In [12]:
"""
MetriplexBridge — Metriplectic Integration Module
======================================================
Golden Operator:
  O_n = cos(π·n) · cos(π·φ·n),  φ ≈ 1.618
  Continuous range O_n ∈ [-1, 1] ↔ discrete_phase_fragment ∈ [0..7]

Connects two layers of the same physical system:
  Big-Endian topological serializer (state pack/unpack)

Metriplectic Architecture:
  d_symp = {u, H}  →  winding=0, ternary=+1  (bosonic, constructive, L_symp)
  d_metr = [u, S]  →  winding=2, ternary=-1  (fermionic, destructive, L_metr)
  Equilibrium       →  ternary=0               (attractor, n=6 in topology_entries)


Author: Jacobo Tlacaelel Mina Rodriguez
"""

import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from typing import Tuple, Dict, List, Callable, Optional
from dataclasses import dataclass, field
from enum import Enum
import warnings
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer

H_MAX = 7
PHI = (1 + math.sqrt(5)) / 2
   


class EnergyProfile(Enum):
    """Predefined energy normalization profiles."""
    LINEAR = "linear"
    QUADRATIC = "quadratic"
    METRIPLEX = "metriplex"
    CUSTOM = "custom"


@dataclass
class MetriplexConfig:
    """Configuration for metriplex oracle."""
    momentum_range: Tuple[int, int] = (1, 6)
    energy_profile: EnergyProfile = EnergyProfile.METRIPLEX
    normalization_target: float = 0.45
    collision_groups: Optional[Dict[str, List[int]]] = None

    def __post_init__(self):
        if self.collision_groups is None:
            self.collision_groups = {
                'A': [1, 2, 3],
                'B': [4, 5, 6]
            }


class MetriplexOracle:
    """
    Metriplex momentum oracle for second quantization.

    Maps momentum states to energy vectors establishing a hidden 2-to-1
    function. Simon's algorithm discovers: momentum pairs differing by 3
    collide. (Symplectic component: phase shifts; Metric component: energy
    relaxation toward normalization_target attractor.)
    """

    def __init__(self, config: MetriplexConfig = None):
        if config is None:
            config = MetriplexConfig()
        self.config = config
        self.p_min, self.p_max = config.momentum_range
        self._build_energy_map()
        self._build_collision_map()

    def _build_energy_map(self):
        self.energy_map = {}
        for p in range(self.p_min, self.p_max + 1):
            if self.config.energy_profile == EnergyProfile.LINEAR:
                raw_energy = p / self.p_max
            elif self.config.energy_profile == EnergyProfile.QUADRATIC:
                raw_energy = (p / self.p_max) ** 2
            elif self.config.energy_profile == EnergyProfile.METRIPLEX:
                alpha = 1.2
                raw_energy = self.config.normalization_target * (p / self.p_max) ** alpha
            else:
                raise ValueError(f"Unknown energy profile: {self.config.energy_profile}")
            self.energy_map[p] = raw_energy

        mean_energy = np.mean([self.energy_map[p] for p in range(self.p_min, self.p_max + 1)])
        if abs(mean_energy - self.config.normalization_target) > 0.01:
            warnings.warn(
                f"Mean energy {mean_energy:.4f} deviates from target "
                f"{self.config.normalization_target:.4f}"
            )

    def _build_collision_map(self):
        self.collision_map = {}
        self.output_groups = {}
        for group_name, momenta in self.config.collision_groups.items():
            for p in momenta:
                self.collision_map[p] = group_name
            group_index = list(self.config.collision_groups.keys()).index(group_name)
            n_groups = len(self.config.collision_groups)
            output_vec = np.zeros(n_groups)
            output_vec[group_index] = 1.0
            self.output_groups[group_name] = output_vec

    def _compute_symmetry_string(self) -> int:
        group_lists = list(self.config.collision_groups.values())
        xor_accumulator = 0
        for group in group_lists:
            for i, p1 in enumerate(group):
                for p2 in group[i+1:]:
                    xor_accumulator |= (p1 ^ p2)
        return xor_accumulator

    def forward(self, momentum: int) -> Tuple[str, np.ndarray, float]:
        """Evaluate oracle at given momentum state."""
        if momentum < self.p_min or momentum > self.p_max:
            raise ValueError(f"Momentum {momentum} out of range [{self.p_min}, {self.p_max}]")
        group = self.collision_map[momentum]
        output_vec = self.output_groups[group]
        energy = self.energy_map[momentum]
        return group, output_vec, energy

    def collide_pair(self, p1: int, p2: int) -> bool:
        return self.collision_map[p1] == self.collision_map[p2]

    def get_collision_partners(self, p: int) -> List[int]:
        group = self.collision_map[p]
        return self.config.collision_groups[group]

    def symmetry_string(self) -> int:
        return self._compute_symmetry_string()

    def to_hilbert_oracle(self, fock_basis) -> Callable:
        """Convert classical oracle to quantum operator for Hilbert space."""
        def quantum_oracle(state_vector: np.ndarray) -> np.ndarray:
            result = state_vector.copy()
            for idx, occupation in enumerate(fock_basis.basis_states):
                effective_momentum = self._occupation_to_momentum(occupation)
                energy = self.energy_map[effective_momentum]
                phase_shift = np.exp(2j * np.pi * energy)
                result[idx] *= phase_shift
            return result
        return quantum_oracle

    def _occupation_to_momentum(self, occupation: Tuple[int, ...]) -> int:
        total_occ = sum(occupation)
        range_size = self.p_max - self.p_min + 1
        p = (total_occ % range_size) + self.p_min
        return p

    def get_oracle_info(self) -> Dict:
        return {
            'momentum_range': self.config.momentum_range,
            'n_groups': len(self.config.collision_groups),
            'collision_groups': {k: v for k, v in self.config.collision_groups.items()},
            'symmetry_string': self.symmetry_string(),
            'energy_profile': self.config.energy_profile.value,
            'normalization_target': self.config.normalization_target,
            'energy_map': {p: self.energy_map[p] for p in range(self.p_min, self.p_max + 1)},
            'collision_structure': {p: self.collision_map[p] for p in range(self.p_min, self.p_max + 1)}
        }


class H7Conservation:
    """
    Mechanism I: H7 Entanglement Conservation

    In 3-qubit Hilbert space (8 basis states 0-7), states are paired
    by the rule: x ↔ (7 ⊕ x). This conservation law is fundamental.
    """

    CONSERVATION_CONSTANT = 7

    @staticmethod
    def partner_state(state: int) -> int:
        if not (0 <= state <= 7):
            raise ValueError("State must be in [0, 7]")
        return H7Conservation.CONSERVATION_CONSTANT ^ state

    @staticmethod
    def verify_pairing(state_a: int, state_b: int) -> bool:
        return state_b == H7Conservation.partner_state(state_a)

    @staticmethod
    def pairing_table() -> Dict[int, int]:
        return {i: H7Conservation.partner_state(i) for i in range(8)}

    @staticmethod
    def verify_conservation_invariant(state_vector: np.ndarray, threshold: float = 1e-6) -> bool:
        if len(state_vector) != 8:
            raise ValueError("State vector must be 8-dimensional (3-qubit Hilbert space)")
        for i in range(8):
            if abs(state_vector[i]) > threshold:
                partner = H7Conservation.partner_state(i)
                if abs(state_vector[partner]) < threshold:
                    return False
        return True
#===========
# Fock
#===========
class OccupationMode(Enum):
    """Enumeration of occupation number modes in Fock space."""
    BOSONIC = "bosonic"
    FERMIONIC = "fermionic"


@dataclass
class FockConfig:
    """Configuration for Fock space instantiation."""
    n_modes: int = 3
    n_max: int = 3
    mode_type: OccupationMode = OccupationMode.BOSONIC
    use_gray_code: bool = True


class FockBasis:
    """
    Fock space representation with second quantization operators.
    dim(H_Fock) = (n_max + 1)^n_modes
    """

    def __init__(self, config: FockConfig = None):
        if config is None:
            config = FockConfig()
        self.config = config
        self.n_modes = config.n_modes
        self.n_max = config.n_max
        self.dim = (config.n_max + 1) ** config.n_modes
        self._build_basis()
        self._precompute_operators()

    def _build_basis(self):
        basis_states = []
        for occupation in np.ndindex(*[self.n_max + 1] * self.n_modes):
            basis_states.append(occupation)
        self.basis_states = np.array(basis_states)
        assert len(self.basis_states) == self.dim
        self.state_to_index = {tuple(state): i for i, state in enumerate(self.basis_states)}
        self.index_to_state = {i: tuple(state) for i, state in enumerate(self.basis_states)}

    def _precompute_operators(self):
        self.creation_ops = {}
        self.annihilation_ops = {}
        for mode in range(self.n_modes):
            self.creation_ops[mode] = self._build_creation_op(mode)
            self.annihilation_ops[mode] = self._build_annihilation_op(mode)

    def _build_creation_op(self, mode: int) -> np.ndarray:
        op = np.zeros((self.dim, self.dim), dtype=complex)
        for i, state in enumerate(self.basis_states):
            state_list = list(state)
            if state_list[mode] < self.n_max:
                new_occupation = state_list[mode] + 1
                amplitude = np.sqrt(new_occupation)
                state_list[mode] = new_occupation
                j = self.state_to_index[tuple(state_list)]
                op[j, i] = amplitude
        return op

    def _build_annihilation_op(self, mode: int) -> np.ndarray:
        op = np.zeros((self.dim, self.dim), dtype=complex)
        for i, state in enumerate(self.basis_states):
            state_list = list(state)
            if state_list[mode] > 0:
                old_occupation = state_list[mode]
                amplitude = np.sqrt(old_occupation)
                state_list[mode] = old_occupation - 1
                j = self.state_to_index[tuple(state_list)]
                op[j, i] = amplitude
        return op

    def get_creation_op(self, mode: int) -> np.ndarray:
        if mode not in self.creation_ops:
            raise ValueError(f"Mode {mode} out of range [0, {self.n_modes-1}]")
        return self.creation_ops[mode].copy()

    def get_annihilation_op(self, mode: int) -> np.ndarray:
        if mode not in self.annihilation_ops:
            raise ValueError(f"Mode {mode} out of range [0, {self.n_modes-1}]")
        return self.annihilation_ops[mode].copy()

    def number_operator(self, mode: int) -> np.ndarray:
        a_dag = self.get_creation_op(mode)
        a = self.get_annihilation_op(mode)
        return a_dag @ a

    def total_number_operator(self) -> np.ndarray:
        N_total = np.zeros((self.dim, self.dim), dtype=complex)
        for mode in range(self.n_modes):
            N_total += self.number_operator(mode)
        return N_total

    def state_vector(self, occupation: Tuple[int, ...]) -> np.ndarray:
        if len(occupation) != self.n_modes:
            raise ValueError(f"Occupation must have {self.n_modes} elements")
        if any(n > self.n_max for n in occupation):
            raise ValueError(f"Occupation exceeds n_max = {self.n_max}")
        vec = np.zeros(self.dim, dtype=complex)
        idx = self.state_to_index[tuple(occupation)]
        vec[idx] = 1.0
        return vec

    def to_gray_code(self, state_index: int) -> int:
        return state_index ^ (state_index >> 1)

    def from_gray_code(self, gray: int) -> int:
        mask = gray
        while mask:
            mask >>= 1
            gray ^= mask
        return gray

    def occupation_to_binary(self, occupation: Tuple[int, ...]) -> str:
        state_index = self.state_to_index[tuple(occupation)]
        if self.config.use_gray_code:
            binary_rep = self.to_gray_code(state_index)
        else:
            binary_rep = state_index
        return format(binary_rep, f'0{self.n_modes * 2}b')

    def get_basis_info(self) -> Dict:
        return {
            'dimension': self.dim,
            'n_modes': self.n_modes,
            'n_max_per_mode': self.n_max,
            'total_occupation_max': self.n_max * self.n_modes,
            'basis_states': self.basis_states.tolist(),
            'use_gray_code': self.config.use_gray_code,
            'mode_type': self.config.mode_type.value
        }


class FockStateVector:
    """Quantum state in Fock basis with helper methods."""

    def __init__(self, fock_basis: FockBasis, vector: np.ndarray = None):
        self.fock = fock_basis
        if vector is None:
            self.vec = np.zeros(fock_basis.dim, dtype=complex)
        else:
            if len(vector) != fock_basis.dim:
                raise ValueError(f"Vector dimension {len(vector)} != {fock_basis.dim}")
            self.vec = vector.astype(complex)

    def normalize(self):
        norm = np.linalg.norm(self.vec)
        if norm > 1e-10:
            self.vec /= norm
        return self

    def occupation_probabilities(self) -> Dict[Tuple[int, ...], float]:
        probs = {}
        for i, state in enumerate(self.fock.basis_states):
            prob = abs(self.vec[i])**2
            if prob > 1e-10:
                probs[tuple(state)] = prob
        return probs

    def expectation_value(self, operator: np.ndarray) -> complex:
        return self.vec @ operator @ self.vec

    def fidelity(self, other: 'FockStateVector') -> float:
        overlap = np.abs(np.vdot(self.vec, other.vec))**2
        return np.clip(overlap, 0, 1)


def fock_ground_state(fock_basis: FockBasis) -> FockStateVector:
    """Return ground state |0, 0, ..., 0⟩."""
    occupation = tuple(0 for _ in range(fock_basis.n_modes))
    vec = fock_basis.state_vector(occupation)
    return FockStateVector(fock_basis, vec)


def fock_single_photon(fock_basis: FockBasis, mode: int) -> FockStateVector:
    """Return single photon in given mode |0, ..., 1_m, ..., 0⟩."""
    occupation = [0] * fock_basis.n_modes
    occupation[mode] = 1
    vec = fock_basis.state_vector(tuple(occupation))
    return FockStateVector(fock_basis, vec)
# ---------------------------------------------------------------------------
# Endian Hexadecimal format
# ---------------------------------------------------------------------------

import numpy as np
from typing import Dict, Tuple, Optional, List, Union
import struct

class HexadecimalSpecifications:
    """Defines exact hexadecimal formats according to user table"""

    FORMATS = {
        'uint8': {
            'bits': 8,
            'bytes': 1,
            'hex_chars': 2,
            'min_value': 0,
            'max_value': 255,
            'signed': False,
            'format': '02X',
            'example': '00'
        },
        'uint16': {
            'bits': 16,
            'bytes': 2,
            'hex_chars': 4,
            'min_value': 0,
            'max_value': 65535,
            'signed': False,
            'format': '04X',
            'example': '0000'
        },
        'uint32': {
            'bits': 32,
            'bytes': 4,
            'hex_chars': 8,
            'min_value': 0,
            'max_value': 4294967295,
            'signed': False,
            'format': '08X',
            'example': '00000000'
        },
        'uint64': {
            'bits': 64,
            'bytes': 8,
            'hex_chars': 16,
            'min_value': 0,
            'max_value': 18446744073709551615,
            'signed': False,
            'format': '016X',
            'example': '0000000000000000'
        },
        'uint128': {
            'bits': 128,
            'bytes': 16,
            'hex_chars': 32,
            'min_value': 0,
            'max_value': 2**128 - 1,
            'signed': False,
            'format': '032X',
            'example': '00000000000000000000000000000000'
        },
        'int8': {
            'bits': 8,
            'bytes': 1,
            'hex_chars': 2,
            'min_value': -128,
            'max_value': 127,
            'signed': True,
            'format': '02X',
            'example': '00'
        },
    }

    @staticmethod
    def validate_value(value: int, type: str) -> bool:
        """Validates that a value is in range for its type"""
        spec = HexadecimalSpecifications.FORMATS[type]
        return spec['min_value'] <= value <= spec['max_value']


# =============================================================================
# ENCODER: Big-Endian Network Byte Order
# =============================================================================

class BigEndianHexadecimalEncoder:
    """
    Encodes values to hexadecimal with big-endian byte ordering.

    Big-endian: MSB first (network order, internet standard)
    Example: 0x1234 -> "1234" (not "3412")
    """

    @staticmethod
    def to_hex_uint8(value: int) -> str:
        assert 0 <= value <= 255, f"uint8 range: 0-255, got {value}"
        return f"{value:02X}"

    @staticmethod
    def to_hex_uint16(value: int) -> str:
        assert 0 <= value <= 65535, f"uint16 range: 0-65535, got {value}"
        return f"{value:04X}"

    @staticmethod
    def to_hex_uint32(value: int) -> str:
        assert 0 <= value <= 4294967295, f"uint32 range: 0-4294967295, got {value}"
        return f"{value:08X}"

    @staticmethod
    def to_hex_uint64(value: int) -> str:
        assert 0 <= value <= 18446744073709551615, f"uint64 range, got {value}"
        return f"{value:016X}"

    @staticmethod
    def to_hex_uint128(value: int) -> str:
        max_128 = (1 << 128) - 1
        assert 0 <= value <= max_128, f"uint128 range, got {value}"
        return f"{value:032X}"

    @staticmethod
    def to_hex_int8(value: int) -> str:
        assert -128 <= value <= 127, f"int8 range: -128 to 127, got {value}"
        if value < 0:
            value = 256 + value
        return f"{value:02X}"

    @staticmethod
    def from_hex_uint8(hex_str: str) -> int:
        value = int(hex_str, 16)
        assert 0 <= value <= 255
        return value

    @staticmethod
    def from_hex_uint16(hex_str: str) -> int:
        value = int(hex_str, 16)
        assert 0 <= value <= 65535
        return value

    @staticmethod
    def from_hex_uint32(hex_str: str) -> int:
        value = int(hex_str, 16)
        assert 0 <= value <= 4294967295
        return value

    @staticmethod
    def from_hex_uint64(hex_str: str) -> int:
        value = int(hex_str, 16)
        assert 0 <= value <= 18446744073709551615
        return value

    @staticmethod
    def from_hex_int8(hex_str: str) -> int:
        value = int(hex_str, 16)
        if value > 127:
            value = value - 256
        assert -128 <= value <= 127
        return value


# =============================================================================
# TOPOLOGICAL ENCODER: Packing + Conversion
# =============================================================================

class TopologicalBigEndianEncoder:
    """
    Packs topology and converts to the exact hexadecimal formats of the table.

    The 7 topology entries mirror the MOD 6 collision structure:
      Group 'a' (winding=0, ternary=+1): x(1,6), y(5,2), z(3,4)  → bosonic / constructive
      Group 'b' (winding=2, ternary=-1): -z(4,3), -y(5,2), -x(6,1) → fermionic / destructive
      Neutral (ternary=0): equilibrium state
    """

    TERNARY_TO_BITS = {
        -1: 0b00,
         0: 0b01,
        +1: 0b10,
    }

    topology_entries = [
        {'index': 1, 'pair': 6, 'winding': 0, 'mapping': 0, 'ternary_weight': 1,  'discrete_phase_fragment': 0},  # n=0: x(1,6)
        {'index': 5, 'pair': 2, 'winding': 0, 'mapping': 0, 'ternary_weight': 1,  'discrete_phase_fragment': 1},  # n=1: y(5,2)
        {'index': 3, 'pair': 4, 'winding': 0, 'mapping': 0, 'ternary_weight': 1,  'discrete_phase_fragment': 6},  # n=2: z(3,4)
        {'index': 4, 'pair': 3, 'winding': 2, 'mapping': 1, 'ternary_weight': -1, 'discrete_phase_fragment': 5},  # n=3: -z(4,3)
        {'index': 5, 'pair': 2, 'winding': 2, 'mapping': 1, 'ternary_weight': -1, 'discrete_phase_fragment': 2},  # n=4: -y(5,2)
        {'index': 6, 'pair': 1, 'winding': 2, 'mapping': 1, 'ternary_weight': -1, 'discrete_phase_fragment': 3},  # n=5: -x(6,1)
        {'index': 2, 'pair': 3, 'winding': 0, 'mapping': 0, 'ternary_weight': 0,  'discrete_phase_fragment': 4},  # n=6: neutral
    ]

    @staticmethod
    def pack_topology(
        index: int,
        pair: int,
        winding: int,
        mapping: int,
        ternary_weight: int,
        discrete_phase_fragment: int,
    ) -> int:
        """Packs topology into 14 bits."""
        assert 1 <= index <= 6, f"Index out of range (1-6): {index}"
        assert 1 <= pair <= 6, f"Pair out of range (1-6): {pair}"
        assert winding in [0, 2], f"Winding must be 0 or 2: {winding}"
        assert mapping in [0, 1], f"Mapping must be 0 or 1: {mapping}"
        assert ternary_weight in [-1, 0, 1], f"Ternary weight must be -1, 0 or 1: {ternary_weight}"
        assert 0 <= discrete_phase_fragment <= 7, f"Discrete phase fragment out of range (0-7): {discrete_phase_fragment}"

        value = 0
        value |= (index - 1) << 0
        value |= (pair - 1) << 3
        value |= (winding // 2) << 6
        value |= mapping << 8
        value |= TopologicalBigEndianEncoder.TERNARY_TO_BITS[ternary_weight] << 9
        value |= discrete_phase_fragment << 11

        return value

    @staticmethod
    def unpack_topology(value: int) -> Dict:
        """Unpacks topological state"""
        index = ((value >> 0) & 0b111) + 1
        pair = ((value >> 3) & 0b111) + 1
        winding = ((value >> 6) & 0b1) * 2
        mapping = (value >> 8) & 0b1

        weight_bits = (value >> 9) & 0b11
        ternary_weight = {0b00: -1, 0b01: 0, 0b10: 1}.get(weight_bits, 0)

        discrete_phase_fragment = (value >> 11) & 0b111
        cycle_phase_derived = (discrete_phase_fragment / 3.5) - 1.0

        return {
            'index': index,
            'pair': pair,
            'winding': winding,
            'mapping': mapping,
            'ternary_weight': ternary_weight,
            'discrete_phase_fragment': discrete_phase_fragment,
            'cycle_phase_derived': cycle_phase_derived,
        }

    @staticmethod
    def generate_random_observable_and_missing_potentials() -> Dict:
        observable_index = np.random.randint(1, 7)
        missing_potential = 7 - observable_index
        discrete_phase_fragment = np.random.randint(0, 8)
        winding = 0
        mapping = 0
        ternary_weight = 1

        packed_value = TopologicalBigEndianEncoder.pack_topology(
            index=observable_index,
            pair=missing_potential,
            winding=winding,
            mapping=mapping,
            ternary_weight=ternary_weight,
            discrete_phase_fragment=discrete_phase_fragment
        )

        return {
            'observable_index': observable_index,
            'missing_potential': missing_potential,
            'discrete_phase_fragment': discrete_phase_fragment,
            'winding': winding,
            'mapping': mapping,
            'ternary_weight': ternary_weight,
            'packed_value': packed_value
        }

    @classmethod
    def complete_hexadecimal_table(cls) -> str:
        table = "\n|- COMPLETE HEXADECIMAL TABLE (Big-Endian)\n"
        table += "|- unsigned 8-bit:   2 digits (00)\n"
        table += "|- unsigned 16-bit:  4 digits (0000)\n"
        table += "|- unsigned 32-bit:  8 digits (00000000)\n"
        table += "|- unsigned 64-bit:  16 digits (0000000000000000)\n"
        table += "|- unsigned 128-bit: 32 digits (00000000000000000000000000000000)\n\n"

        table += "n | Index | Pair | Winding | Mapping | Weight | Discrete_phase | uint8 | uint16 | uint32   | uint64\n"
        table += "-"*110 + "\n"

        for n in range(len(cls.topology_entries)):
            entry = cls.topology_entries[n]
            value = cls.pack_topology(
                index=entry['index'],
                pair=entry['pair'],
                winding=entry['winding'],
                mapping=entry['mapping'],
                ternary_weight=entry['ternary_weight'],
                discrete_phase_fragment=entry['discrete_phase_fragment']
            )

            hex8  = BigEndianHexadecimalEncoder.to_hex_uint8(value & 0xFF)
            hex16 = BigEndianHexadecimalEncoder.to_hex_uint16(value & 0xFFFF)
            hex32 = BigEndianHexadecimalEncoder.to_hex_uint32(value & 0xFFFFFFFF)
            hex64 = BigEndianHexadecimalEncoder.to_hex_uint64(value & 0xFFFFFFFFFFFFFFFF)

            table += (
                f"{n} |   {entry['index']}    |   {entry['pair']}    |   {entry['winding']}     |   "
                f"{entry['mapping']}   |  {entry['ternary_weight']:>2}  |   {entry['discrete_phase_fragment']:>7} | "
                f"{hex8:>5} | {hex16:>6} | {hex32:>8} | {hex64}\n"
            )

        return table


# =============================================================================
# SERIALIZER: Bytes in Big-Endian
# =============================================================================

class BigEndianSerializer:
    """Serializes states respecting big-endian byte ordering"""

    @staticmethod
    def value_to_bytes_uint16(value: int, endian: str = 'big') -> bytes:
        return struct.pack(f'>H', value)

    @staticmethod
    def value_to_bytes_uint32(value: int, endian: str = 'big') -> bytes:
        return struct.pack(f'>I', value)

    @staticmethod
    def value_to_bytes_uint64(value: int, endian: str = 'big') -> bytes:
        return struct.pack(f'>Q', value)

    @staticmethod
    def bytes_to_value_uint16(data: bytes, endian: str = 'big') -> int:
        return struct.unpack(f'>H', data)[0]


# =============================================================================
# DEMONSTRATION: IMPROVED MOD 6
# =============================================================================

def demo_improved_mod6():
    """Demonstrates MOD 6 with exact hexadecimal formats"""

    print("\n" + "#"*80)
    print("  IMPROVED MOD 6: EXACT BIG-ENDIAN HEXADECIMAL")
    print("#"*80)

    print(TopologicalBigEndianEncoder.complete_hexadecimal_table())

    print("\n[DETAILED EXAMPLE: n=0 (Index 1, Pair 6)]")
    print("-"*80)

    n_example = 0
    example_entry = TopologicalBigEndianEncoder.topology_entries[n_example]
    example_discrete_phase_fragment = example_entry['discrete_phase_fragment']
    example_cycle_phase_derived = (example_discrete_phase_fragment / 3.5) - 1.0

    value = TopologicalBigEndianEncoder.pack_topology(
        index=example_entry['index'],
        pair=example_entry['pair'],
        winding=example_entry['winding'],
        mapping=example_entry['mapping'],
        ternary_weight=example_entry['ternary_weight'],
        discrete_phase_fragment=example_discrete_phase_fragment
    )

    print(f"\nPacked integer value: {value} (decimal)")
    print(f"Packed integer value: 0x{value:04X} (hexadecimal)")
    print(f"Discrete phase fragment (packed): {example_discrete_phase_fragment}")
    print(f"Derived cycle phase (from discrete fragment): {example_cycle_phase_derived}")

    print(f"\nConversion to formats from the table:")
    print(f"  unsigned 8-bit   (2 digits):   {BigEndianHexadecimalEncoder.to_hex_uint8(value & 0xFF)}")
    print(f"  unsigned 16-bit  (4 digits):   {BigEndianHexadecimalEncoder.to_hex_uint16(value & 0xFFFF)}")
    print(f"  unsigned 32-bit  (8 digits):   {BigEndianHexadecimalEncoder.to_hex_uint32(value & 0xFFFFFFFF)}")
    print(f"  unsigned 64-bit  (16 digits):  {BigEndianHexadecimalEncoder.to_hex_uint64(value & 0xFFFFFFFFFFFFFFFF)}")
    print(f"  unsigned 128-bit (32 digits):  {BigEndianHexadecimalEncoder.to_hex_uint128(value & ((1 << 128) - 1))}")

    print(f"\n[SIGNED INTEGERS (Two's Complement)]")
    print("-"*80)
    for signed_value, description in [(0, "0 positive"), (127, "127 max positive"), (-1, "-1"), (-128, "-128 min negative")]:
        print(f"  {signed_value:4d} ({description:25s}) -> {BigEndianHexadecimalEncoder.to_hex_int8(signed_value)}")

    print(f"\n[BIG-ENDIAN vs LITTLE-ENDIAN]")
    print("-"*80)
    example_value = 0x3028
    print(f"\nValue: 0x{example_value:04X} (decimal: {example_value})")
    bytes_be = struct.pack('>H', example_value)
    print(f"  Bytes (big-endian):    {bytes_be.hex().upper()}")
    bytes_le = struct.pack('<H', example_value)
    print(f"  Bytes (little-endian): {bytes_le.hex().upper()} (DIFFERENT)")

    print(f"\n[RECOVERY (Unpacking)]")
    print("-"*80)
    packed_value_for_unpacking = TopologicalBigEndianEncoder.pack_topology(
        index=example_entry['index'],
        pair=example_entry['pair'],
        winding=example_entry['winding'],
        mapping=example_entry['mapping'],
        ternary_weight=example_entry['ternary_weight'],
        discrete_phase_fragment=example_discrete_phase_fragment
    )
    hex_packed = f"{packed_value_for_unpacking:04X}"
    recovered_value = BigEndianHexadecimalEncoder.from_hex_uint16(hex_packed)
    topo = TopologicalBigEndianEncoder.unpack_topology(recovered_value)

    print(f"\nPacked hexadecimal: {hex_packed}")
    print(f"Recovered integer value: {recovered_value}")
    print(f"Unpacked topology:")
    for k, v in topo.items():
        print(f"  {k}: {v}")

    print(f"\n[RANDOM OBSERVABLE GENERATION AND MISSING POTENTIALS]")
    print("-"*80)
    random_observable_data = TopologicalBigEndianEncoder.generate_random_observable_and_missing_potentials()
    for k, v in random_observable_data.items():
        print(f"  {k}: {v}")
    print(f"  Packed value (hexadecimal): {BigEndianHexadecimalEncoder.to_hex_uint16(random_observable_data['packed_value'])}")

    print(f"\n[EXACT FORMAT SPECIFICATIONS]")
    print("-"*80)
    for type_name, spec in HexadecimalSpecifications.FORMATS.items():
        print(f"\n{type_name.upper()}:")
        print(f"  Bits: {spec['bits']}, Bytes: {spec['bytes']}, Hex digits: {spec['hex_chars']}")
        print(f"  Range: {spec['min_value']} to {spec['max_value']}")

    print("\n" + "#"*80)


if __name__ == '__main__':
    demo_improved_mod6()
# ---------------------------------------------------------------------------
# Physical constants
# ---------------------------------------------------------------------------
PHI: float = (1.0 + (math.sqrt(5.0))) / 2.0   # Golden ratio φ ≈ 1.618
PI: float = math.pi

# Theoretical ratio Γ_π / Γ_φ = φ² ≈ 2.618 (Mandate Postulate)
GOLDEN_RATIO_SQUARED: float = PHI ** 2


# ---------------------------------------------------------------------------
# Golden Operator Utilities
# ---------------------------------------------------------------------------

def golden_operator(n: int, phi: float = PHI) -> float:
    """
    Structured Golden Operator (Rule 2.1):
      O_n = cos(π·n) · cos(π·φ·n)

    Range: O_n ∈ [-1, 1]
    - Polarity π  →  Particle tendency / wave collapse
    - Laminar π/φ →  Wave tendency / coherent flow
    """
    parity = math.cos(PI * n)
    quasiperiod = math.cos(PI * phi * n)
    return parity * quasiperiod


def o_n_to_phase_fragment(o_n_value: float) -> int:
    """
    Maps O_n ∈ [-1, 1] → discrete_phase_fragment ∈ [0..7].

    Inversion of: cycle_phase_derived = (fragment / 3.5) - 1.0
    Therefore:    fragment = round((o_n_value + 1.0) * 3.5)
    Clamped to [0, 7] to guarantee validity in pack_topology.
    """
    fragment = round((o_n_value + 1.0) * 3.5)
    return int(np.clip(fragment, 0, 7))


def phase_fragment_to_o_n(fragment: int) -> float:
    """Exact inversion: fragment ∈ [0..7] → O_n ∈ [-1.0, 1.0]."""
    return (fragment / 3.5) - 1.0


# ---------------------------------------------------------------------------
# Main Class: MetriplexEndianBridge
# ---------------------------------------------------------------------------

class MetriplexEndianBridge:
    """
    Bridge between the quantum engine (h_logic) and the serializer (endian).

    Complete flow of a state:
      Fock |n0,n1,n2⟩
        → MetriplexOracle → (group, energy, vector_output)
        → H7Conservation  → h7_state and its partner
        → golden_operator → O_n → discrete_phase_fragment
        → TopologicalBigEndianEncoder.pack_topology → 14-bit integer
        → BigEndianHexadecimalEncoder → hex string (uint8/16/32/64/128)
    """

    def __init__(
        self,
        oracle: Optional[MetriplexOracle] = None,
        fock: Optional[FockBasis] = None,
    ):
        self.oracle = oracle or MetriplexOracle(MetriplexConfig())
        self.fock = fock or FockBasis(FockConfig(n_modes=3, n_max=2))
        self._encoder = TopologicalBigEndianEncoder
        self._hex = BigEndianHexadecimalEncoder

    # ------------------------------------------------------------------
    # Lagrangian: Symplectic and Metric components
    # ------------------------------------------------------------------

    def compute_lagrangian(self) -> Tuple[float, float]:
        """
        Metriplectic Lagrangian calculation (Rule 3.1):

          L_symp = Σ energy of entries with winding=0  (conservative, {u,H})
          L_metr = Σ energy of entries with winding=2  (dissipative,  [u,S])

        Invariant (Rule 1.3): L_symp > 0 and L_metr > 0 always.
        Stationary ratio:     L_symp / L_metr ≈ φ² ≈ 2.618

        Returns:
            (L_symp, L_metr)
        """
        L_symp = 0.0
        L_metr = 0.0

        for entry in self._encoder.topology_entries:
            p = entry['index']
            if p < self.oracle.p_min or p > self.oracle.p_max:
                continue
            _, _, energy = self.oracle.forward(p)

            if entry['winding'] == 0:
                L_symp += energy
            elif entry['winding'] == 2:
                L_metr += energy

        return L_symp, L_metr

    # ------------------------------------------------------------------
    # Encoding: Fock → hex
    # ------------------------------------------------------------------

    def encode_fock_state(self, occupation: Tuple[int, ...], fmt: str = 'uint16') -> str:
        """
        Encodes a Fock state to hexadecimal using the topological encoder.

        Steps:
          1. occupation → momentum p (MetriplexOracle._occupation_to_momentum)
          2. p → group + energy (oracle.forward)
          3. group A → winding=0, ternary=+1; group B → winding=2, ternary=-1
          4. p → H7 state (p-1) → discrete_phase_fragment via O_{p-1}
          5. pack_topology → integer → hex

        Args:
            occupation: Tuple of Fock occupations (n0, n1, n2)
            fmt: hex format ('uint8','uint16','uint32','uint64','uint128')

        Returns:
            Big-Endian hexadecimal string
        """
        p = self.oracle._occupation_to_momentum(occupation)
        group, _, energy = self.oracle.forward(p)

        # Topological parameters from group membership
        if group == 'A':
            winding = 0
            mapping = 0
            ternary = 1
        else:  # group B
            winding = 2
            mapping = 1
            ternary = -1

        # H7 state: use (p-1) as 3-qubit state index [0..7]
        h7_state = (p - 1) % 8
        o_n_val = golden_operator(h7_state)
        phase_frag = o_n_to_phase_fragment(o_n_val)

        packed = self._encoder.pack_topology(
            index=p,
            pair=7 - p if 1 <= 7 - p <= 6 else p,
            winding=winding,
            mapping=mapping,
            ternary_weight=ternary,
            discrete_phase_fragment=phase_frag,
        )

        return self._to_hex(packed, fmt)

    def _to_hex(self, value, fmt: str) -> str:
        masks = {
            'uint8':   (0xFF,                   self._hex.to_hex_uint8),
            'uint16':  (0xFFFF,                 self._hex.to_hex_uint16),
            'uint32':  (0xFFFFFFFF,             self._hex.to_hex_uint32),
            'uint64':  (0xFFFFFFFFFFFFFFFF,     self._hex.to_hex_uint64),
            'uint128': ((1 << 128) - 1,         self._hex.to_hex_uint128),
        }
        if fmt not in masks:
            raise ValueError(f"Unknown format: {fmt}. Choose from {list(masks)}")
        mask, encoder = masks[fmt]
        return encoder(int(value) & mask)  # int() cast avoids numpy ufunc error

    # ------------------------------------------------------------------
    # Decoding: hex → recovered state
    # ------------------------------------------------------------------

    def decode_hex_state(self, hex_str: str) -> Dict:
        """
        Decodes a hexadecimal state (uint16) to the full topological state.

        Returns:
            Dict containing: index, pair, winding, mapping, ternary_weight,
                             discrete_phase_fragment, cycle_phase_derived,
                             group, energy, h7_state, h7_partner, o_n
        """
        raw = self._hex.from_hex_uint16(hex_str)
        topo = self._encoder.unpack_topology(raw)

        p = topo['index']
        group, _, energy = self.oracle.forward(
            max(self.oracle.p_min, min(self.oracle.p_max, p))
        )

        h7_state = (p - 1) % 8
        h7_partner = H7Conservation.partner_state(h7_state)
        o_n_val = phase_fragment_to_o_n(topo['discrete_phase_fragment'])

        return {
            **topo,
            'group': group,
            'energy': energy,
            'h7_state': h7_state,
            'h7_partner': h7_partner,
            'o_n': o_n_val,
        }

    # ------------------------------------------------------------------
    # Conservation Verifications
    # ------------------------------------------------------------------

    def verify_h7_roundtrip(self, h7_state: int) -> bool:
        """
        Verifies that H7 conservation is preserved through hex encoding.

        Strategy: h7_state ∈ [0..7] is encoded DIRECTLY as
        discrete_phase_fragment (both live in the same [0..7] range).
        Thus the roundtrip pack→hex→unpack is exact for all 8 states.

        Limits Test (Rule 3): valid for all x ∈ [0..7].
        """
        if not (0 <= h7_state <= 7):
            raise ValueError("h7_state must be in [0..7]")

        # Map h7_state [0..7] to valid momentum [1..6] via mod 6
        p = (h7_state % 6) + 1
        pair = 7 - p if 1 <= 7 - p <= 6 else p
        group, _, _ = self.oracle.forward(p)
        winding = 0 if group == 'A' else 2
        ternary = 1 if group == 'A' else -1
        mapping = 0 if group == 'A' else 1

        # Encode h7_state *directly* as discrete_phase_fragment — no O_n rounding loss
        packed = self._encoder.pack_topology(
            index=p,
            pair=pair,
            winding=winding,
            mapping=mapping,
            ternary_weight=ternary,
            discrete_phase_fragment=h7_state,
        )
        hex_str = self._hex.to_hex_uint16(int(packed) & 0xFFFF)

        # Decode and verify exact recovery
        raw = self._hex.from_hex_uint16(hex_str)
        topo = self._encoder.unpack_topology(raw)
        recovered_h7 = topo['discrete_phase_fragment']

        expected_partner = H7Conservation.partner_state(h7_state)
        actual_partner = H7Conservation.partner_state(recovered_h7)
        return recovered_h7 == h7_state and actual_partner == expected_partner

    def verify_all_h7_states(self) -> Dict[int, bool]:
        """Executes verify_h7_roundtrip for all 8 H7 states."""
        return {s: self.verify_h7_roundtrip(s) for s in range(8)}

    # ------------------------------------------------------------------
    # Full State Report
    # ------------------------------------------------------------------

    def full_state_report(self, occupation: Tuple[int, ...]) -> Dict:
        """
        Complete report of the Fock state:
          occupation → oracle → H7 → O_n → hex → L_symp, L_metr
        """
        p = self.oracle._occupation_to_momentum(occupation)
        group, output_vec, energy = self.oracle.forward(p)
        h7_state = (p - 1) % 8
        h7_partner = H7Conservation.partner_state(h7_state)
        o_n_val = golden_operator(h7_state)
        phase_frag = o_n_to_phase_fragment(o_n_val)

        hex16 = self.encode_fock_state(occupation, fmt='uint16')
        hex64 = self.encode_fock_state(occupation, fmt='uint64')

        L_symp, L_metr = self.compute_lagrangian()
        ratio = L_symp / L_metr if L_metr > 0 else float('inf')

        return {
            'occupation':    occupation,
            'momentum_p':    p,
            'group':         group,
            'output_vector': output_vec.tolist(),
            'energy':        energy,
            'h7_state':      h7_state,
            'h7_partner':    h7_partner,
            'o_n':           o_n_val,
            'phase_frag':    phase_frag,
            'hex_uint16':    hex16,
            'hex_uint64':    hex64,
            'L_symp':        L_symp,
            'L_metr':        L_metr,
            'L_ratio':       ratio,
            'golden_target': GOLDEN_RATIO_SQUARED,
            'ratio_ok':      abs(ratio - GOLDEN_RATIO_SQUARED) < 1.0,
        }

    # ------------------------------------------------------------------
    # Visual Diagnostics (Rule 3.3)
    # ------------------------------------------------------------------

    def visualize_competition(self, show: bool = True) -> plt.Figure:
        """
        Diagnostic plot: L_symp vs L_metr competition (Rule 3.3).

        Shows for each topological entry:
          - Energy of the entry
          - winding/ternary classification
          - O_n operator
          - Accumulated L_symp and L_metr
        """
        entries = self._encoder.topology_entries
        n_entries = len(entries)

        labels = []
        energies = []
        ternaries = []
        o_n_values = []
        colors = []
        cumul_symp = []
        cumul_metr = []
        acc_s = 0.0
        acc_m = 0.0

        color_map = {1: '#4fc3f7', -1: '#ef5350', 0: '#a5d6a7'}

        for entry in entries:
            p = entry['index']
            p_clamped = max(self.oracle.p_min, min(self.oracle.p_max, p))
            _, _, energy = self.oracle.forward(p_clamped)

            h7_state = (p - 1) % 8
            o_n = golden_operator(h7_state)
            tw = entry['ternary_weight']

            if entry['winding'] == 0:
                acc_s += energy
            elif entry['winding'] == 2:
                acc_m += energy

            labels.append(
                f"n={entries.index(entry)}\n"
                f"p={p}, {'A' if entry['mapping']==0 else 'B'}\n"
                f"w={entry['winding']}"
            )
            energies.append(energy)
            ternaries.append(tw)
            o_n_values.append(o_n)
            colors.append(color_map[tw])
            cumul_symp.append(acc_s)
            cumul_metr.append(acc_m)

        x = np.arange(n_entries)

        fig = plt.figure(figsize=(14, 10), facecolor='#0d1117')
        fig.suptitle(
            'MetriplexBridge — Symplectic/Metric Competition\n'
            r'$\mathcal{L}_{symp}=\{u,H\}$ vs $\mathcal{L}_{metr}=[u,S]$',
            color='white', fontsize=14, fontweight='bold'
        )

        gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

        ax_style = dict(facecolor='#161b22', tick_params={'colors': 'white'},
                        label_color='white')

        # Panel 1 — Energy per entry with ternary color
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.set_facecolor('#161b22')
        bars = ax1.bar(x, energies, color=colors, edgecolor='#30363d', linewidth=0.8)
        ax1.set_title('Energy per topological entry', color='white', fontsize=11)
        ax1.set_xticks(x)
        ax1.set_xticklabels(labels, color='#8b949e', fontsize=7)
        ax1.tick_params(colors='white')
        ax1.set_ylabel('Normalized energy', color='white')
        ax1.spines[:].set_color('#30363d')
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='#4fc3f7', label='Constructive (ternary=+1, winding=0)'),
            Patch(facecolor='#ef5350', label='Destructive (ternary=-1, winding=2)'),
            Patch(facecolor='#a5d6a7', label='Equilibrium (ternary=0)'),
        ]
        ax1.legend(handles=legend_elements, fontsize=7, facecolor='#0d1117',
                   labelcolor='white', edgecolor='#30363d')

        # Panel 2 — O_n per entry
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.set_facecolor('#161b22')
        ax2.axhline(0, color='#30363d', linewidth=0.8, linestyle='--')
        ax2.fill_between(x, 0, o_n_values, where=[v > 0 for v in o_n_values],
                          color='#4fc3f7', alpha=0.6, label='O_n > 0 (wave)')
        ax2.fill_between(x, 0, o_n_values, where=[v <= 0 for v in o_n_values],
                          color='#ef5350', alpha=0.6, label='O_n ≤ 0 (particle)')
        ax2.plot(x, o_n_values, 'o-', color='white', markersize=5, linewidth=1.5)
        ax2.set_title(r'Golden Operator $O_n = \cos(\pi n)\cdot\cos(\pi\phi n)$',
                      color='white', fontsize=11)
        ax2.set_xticks(x)
        ax2.set_xticklabels([f"n={i}" for i in range(n_entries)], color='#8b949e', fontsize=8)
        ax2.tick_params(colors='white')
        ax2.set_ylabel('O_n', color='white')
        ax2.spines[:].set_color('#30363d')
        ax2.legend(fontsize=8, facecolor='#0d1117', labelcolor='white', edgecolor='#30363d')

        # Panel 3 — Accumulated L_symp vs L_metr
        ax3 = fig.add_subplot(gs[1, 0])
        ax3.set_facecolor('#161b22')
        ax3.plot(x, cumul_symp, 's-', color='#4fc3f7', linewidth=2, markersize=7,
                 label=r'$\mathcal{L}_{symp}$ (winding=0, conservative)')
        ax3.plot(x, cumul_metr, '^-', color='#ef5350', linewidth=2, markersize=7,
                 label=r'$\mathcal{L}_{metr}$ (winding=2, dissipative)')
        # Target φ² ratio
        L_s, L_m = cumul_symp[-1], cumul_metr[-1]
        if L_m > 0:
            ratio = L_s / L_m
            ax3.set_title(
                f'Accumulated: L_symp/L_metr = {ratio:.3f}  '
                f'(target φ²={GOLDEN_RATIO_SQUARED:.3f})',
                color='white', fontsize=10
            )
        else:
            ax3.set_title('Accumulated Lagrangian', color='white', fontsize=11)
        ax3.set_xticks(x)
        ax3.set_xticklabels([f"n={i}" for i in range(n_entries)], color='#8b949e', fontsize=8)
        ax3.tick_params(colors='white')
        ax3.set_ylabel('Accumulated energy', color='white')
        ax3.spines[:].set_color('#30363d')
        ax3.legend(fontsize=8, facecolor='#0d1117', labelcolor='white', edgecolor='#30363d')

        # Panel 4 — Hex table for the 7 states
        ax4 = fig.add_subplot(gs[1, 1])
        ax4.set_facecolor('#161b22')
        ax4.axis('off')

        table_data = []
        col_labels = ['n', 'p', 'Ternary', 'O_n', 'Hex16']
        for i, entry in enumerate(entries):
            p = entry['index']
            p_clamped = max(self.oracle.p_min, min(self.oracle.p_max, p))
            _, _, energy = self.oracle.forward(p_clamped)
            h7_state = (p - 1) % 8
            o_n = golden_operator(h7_state)
            phase_frag = o_n_to_phase_fragment(o_n)
            pair = 7 - p if 1 <= 7 - p <= 6 else p
            tw = entry['ternary_weight']
            winding = entry['winding']
            mapping = entry['mapping']
            packed = self._encoder.pack_topology(
                index=p, pair=pair, winding=winding,
                mapping=mapping, ternary_weight=tw,
                discrete_phase_fragment=phase_frag
            )
            hex16 = self._hex.to_hex_uint16(packed & 0xFFFF)
            table_data.append([str(i), str(p), f"{tw:+d}", f"{o_n:.3f}", hex16])

        tbl = ax4.table(
            cellText=table_data,
            colLabels=col_labels,
            cellLoc='center',
            loc='center',
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        for (r, c), cell in tbl.get_celld().items():
            cell.set_facecolor('#0d1117' if r == 0 else '#161b22')
            cell.set_edgecolor('#30363d')
            cell.set_text_props(color='white' if r > 0 else '#58a6ff')
        ax4.set_title('Encoded topological states (uint16)', color='white', fontsize=10)

        if show:
            plt.show()
        return fig


# ---------------------------------------------------------------------------
# Main Demonstration Script
# ---------------------------------------------------------------------------

def main():
    print("\n" + "="*70)
    print("  MetriplexBridge — Integration Demonstration")
    print("="*70)

    bridge = MetriplexEndianBridge()

    # 1. Lagrangian
    L_symp, L_metr = bridge.compute_lagrangian()
    ratio = L_symp / L_metr if L_metr > 0 else float('inf')
    print(f"\n[1] Metriplectic Lagrangian:")
    print(f"  L_symp (symplectic, {{u,H}}): {L_symp:.6f}")
    print(f"  L_metr (metric,    [u,S]):  {L_metr:.6f}")
    print(f"  Ratio L_symp/L_metr:         {ratio:.4f}")
    print(f"  Target φ² ≈ {GOLDEN_RATIO_SQUARED:.4f}  |  Δ = {abs(ratio - GOLDEN_RATIO_SQUARED):.4f}")

    # 2. Golden Operator O_n for the 7 entries
    print(f"\n[2] Golden Operator O_n = cos(πn)·cos(πφn):")
    for i in range(7):
        o_n = golden_operator(i)
        frag = o_n_to_phase_fragment(o_n)
        recovered = phase_fragment_to_o_n(frag)
        print(f"  n={i}: O_n={o_n:+.6f}  →  fragment={frag}  →  O_n_recovered={recovered:+.6f}")

    # 3. Fock state encoding
    print(f"\n[3] Fock Encoding → Hex (Big-Endian):")
    test_states = [(0, 0, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1), (1, 1, 0), (2, 0, 1)]
    for occ in test_states:
        for fmt in ('uint8', 'uint16', 'uint64'):
            hex_val = bridge.encode_fock_state(occ, fmt=fmt)
            print(f"  |{occ}⟩ → {fmt}: {hex_val}")
        print()

    # 4. Round-trip decoding
    print(f"\n[4] Round-trip: Fock → uint16 → decode:")
    occ_test = (1, 0, 2)
    hex16 = bridge.encode_fock_state(occ_test, fmt='uint16')
    decoded = bridge.decode_hex_state(hex16)
    print(f"  Occupation: {occ_test}")
    print(f"  Hex uint16: {hex16}")
    print(f"  Decoded:")
    for k, v in decoded.items():
        print(f"    {k}: {v}")

    # 5. Full H7 Verification
    print(f"\n[5] H7 Conservation Verification (for all 8 states):")
    results = bridge.verify_all_h7_states()
    all_ok = all(results.values())
    for state, ok in results.items():
        partner = H7Conservation.partner_state(state)
        status = "✓" if ok else "✗"
        print(f"  |{state}⟩ ↔ |{partner}⟩  [{status}]")
    print(f"  Global H7 Conservation: {'PRESERVED ✓' if all_ok else 'VIOLATED ✗'}")

    # 6. Full state report
    print(f"\n[6] Complete report for |1,0,0⟩:")
    report = bridge.full_state_report((1, 0, 0))
    for k, v in report.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.6f}")
        else:
            print(f"  {k}: {v}")

    # 7. Visual diagnostics
    print(f"\n[7] Generating diagnostic visualization...")
    bridge.visualize_competition(show=True)

    print("\n" + "="*70)
    print("  Integration completed — Level 3 Physical Isomorphism verified")
    print("="*70 + "\n")


# === NUEVA CAPA LETE: Little-Endian Ternary Exclusion ===
class LETEEncoder:
    def __init__(self):
        self.quasiperiod = 0.36237
        self.golden_ratio = (1 + np.sqrt(5)) / 2  # φ

    def ternary_group(self, value: int, digits: int = 3):
        """Convierte un entero a grupos ternarios little-endian (base-3)"""
        ternary = []
        while value > 0 or len(ternary) < digits:
            ternary.append(value % 3)
            value //= 3
        return ternary[::-1] if len(ternary) > digits else ternary  # little-endian: bajo primero? Wait, ajustamos orden

    def pauli_xor_exclusion(self, state_a: np.ndarray, state_b: np.ndarray):
        """XOR como principio de exclusión: no pueden ser idénticos"""
        xor_result = np.bitwise_xor(state_a, state_b)
        if np.all(xor_result == 0):  # mismo estado → violación Pauli
            return None  # rechazar o aplicar corrección Ca²⁺/K⁺
        return xor_result

    def pack_complex(self, pool_id: int, spike_pattern: list, level: int = 0):
        """Empaqueta con grupos de 3 + residuo áureo"""
        packed = 0
        for i, spike in enumerate(spike_pattern[:9]):  # ejemplo 9 spikes → 3 grupos de 3
            group = self.ternary_group(spike, 3)
            for j, digit in enumerate(group):
                packed = (packed << 2) | digit  # shift + ternario empaquetado
        
        # Inyectar quasiperiod como fracción .1 áurea
        fractional = int(self.quasiperiod * (self.golden_ratio ** level) * 1000) % 1024
        packed = (packed << 10) | fractional
        
        return packed  # little-endian style empaquetado

    def apply_exclusion(self, pool_state, partner_state):
        """Aplica XOR-Pauli entre pool y su mirror H7"""
        a = np.array([int(pool_state['packed'], 16) % 8])   # ejemplo simplificado
        b = np.array([int(partner_state['packed'], 16) % 8])
        result = self.pauli_xor_exclusion(a, b)
    
      # Aquí se tiene que llamar el circuito stim K⁺
         # Lazy Qiskit imports — only needed when running as script
    print("Generating circuit draw...")
    qc_demo.draw(output='text')
    plt.show()

    print("Generating Bloch`s Spheres...")
    # plot_bloch_multivector(qc_demo)
    plt.show()

    qc_demo.barrier()
    qc_demo.measure([0, 1, 2], [1, 0, 2])

    print("Final measured circuit:")
    qc_demo.draw(output='text')
    plt.show()

    from qiskit import QuantumCircuit
    from qiskit.visualization import plot_histogram, plot_bloch_multivector
    from qiskit.quantum_info import Statevector
    import matplotlib.pyplot as plt

    # --- Configuración y Constantes ---
    phi = (1 + np.sqrt(5)) / 2
    qc_demo = QuantumCircuit(3, 3)

    qc_demo.h([0, 1, 2])
    qc_demo.rz(math.cos(phi), 0)
    qc_demo.ry(math.cos(np.pi), 0)
    qc_demo.rx(0, 0)
    qc_demo.cswap(0, 2, 1)
    qc_demo.ccx(2, 1, 0)

    print("Generating circuit draw...")
    qc_demo.draw(output='text')
    plt.show()

    psi_demo = Statevector.from_instruction(qc_demo)

    print("Generating Bloch`s Spheres...")
    plot_bloch_multivector(psi_demo)
    plt.show()

    print("Generating probabilities histogram...")
    probs_dict_demo = psi_demo.probabilities_dict()
    plot_histogram(probs_dict_demo, title="Distribución de Probabilidades del Estado")
    plt.show()

    qc_demo.barrier()
    qc_demo.measure([0, 1, 2], [1, 0, 2])

    print("Final measured circuit:")
    qc_demo.draw(output='text')
    plt.show()

    print(f"\nOperations count: {qc_demo.count_ops()}")
    for i in range(3):
        print(f"Probabilities per Qubit {i}: {psi_demo.probabilities([i])}")

    # MetriplexOracle demo
    config = MetriplexConfig()
    oracle = MetriplexOracle(config)

    print("\nMetriplex Oracle Information:")
    info = oracle.get_oracle_info()
    print(f"  Momentum range: {info['momentum_range']}")
    print(f"  Hidden symmetry string s = {info['symmetry_string']}")

    print(f"\nEnergy Map (normalized):")
    for p in range(1, 7):
        group, output, energy = oracle.forward(p)
        print(f"  p={p}: group={group}, energy={energy:.4f}, output={output}")

    print(f"\nH7 Conservation Pairing:")
    pairing = H7Conservation.pairing_table()
    for state, partner in pairing.items():
        print(f"  |{state}⟩ ↔ |{partner}⟩")

    # FockBasis demo
    fock_config = FockConfig(n_modes=3, n_max=2)
    fock = FockBasis(fock_config)
    print(f"\nFock Space Dimension: {fock.dim}")

    # Main experiment
    n_value = 1
    phi_value = (1 + math.sqrt(5)) / 2
    # run_quantum_experiment(n_value, phi_value)
    # MetriplexOracle demo
    config = MetriplexConfig()
    oracle = MetriplexOracle(config)

    print("\nMetriplex Oracle Information:")
    info = oracle.get_oracle_info()
    print(f"  Momentum range: {info['momentum_range']}")
    print(f"  Hidden symmetry string s = {info['symmetry_string']}")

    print(f"\nEnergy Map (normalized):")
    for p in range(1, 7):
        group, output, energy = oracle.forward(p)
        print(f"  p={p}: group={group}, energy={energy:.4f}, output={output}")

    print(f"\nH7 Conservation Pairing:")
    pairing = H7Conservation.pairing_table()
    for state, partner in pairing.items():
        print(f"  |{state}⟩ ↔ |{partner}⟩")

    # FockBasis demo
    fock_config = FockConfig(n_modes=3, n_max=2)
    fock = FockBasis(fock_config)
    print(f"\nFock Space Dimension: {fock.dim}")

    # Main experiment
    n_value = 1
    phi_value = (1 + math.sqrt(5)) / 2
    # run_quantum_experiment(n_value, phi_value)
    print(fock_config)


bert = SentenceTransformer('all-MiniLM-L6-v2')
def hybrid_embedding(text):
    h7_emb = model(text, endian="little").squeeze(0)  # [128]
    bert_emb = torch.tensor(bert.encode(text))         # [384]
    combined = torch.cat([bert_emb, h7_emb])           # [512]
    return combined

class H7Embedding(nn.Module):
    """
    Embedding layer que combina la métrica H7 con endianness y UTF-8.
    Versión corregida y funcional.
    """
    def __init__(self, embed_dim: int = 64, window: int = 4):
        super().__init__()
        self.embed_dim = embed_dim
        self.window = window
        
        # Proyección de 4 features H7 → embed_dim
        self.proj = nn.Linear(4, embed_dim)

    def byte_to_z7(self, b: int) -> int:
        """Mapea un byte (0-255) a Z7 (1-7)"""
        return (b % 6) + 1 if b != 0 else 6

    def text_to_z7(self, text: str, endian: str = "little") -> torch.Tensor:
        """Convierte texto a secuencia Z7"""
        bytes_data = text.encode('utf-8')
        if endian == "big":
            bytes_data = bytes_data[::-1]
        seq = [self.byte_to_z7(b) for b in bytes_data]
        return torch.tensor(seq, dtype=torch.long)

    def compute_h7_features(self, seq: torch.Tensor) -> torch.Tensor:
        """Calcula features H7 por ventana deslizante"""
        seq_np = seq.cpu().numpy()
        L = len(seq_np)
        
        # Inicializar lista de features
        features = []
        
        # Si la secuencia es más corta que la ventana, usar solo una ventana
        if L < self.window:
            windows = [seq_np]
        else:
            windows = [seq_np[i:i+self.window] for i in range(L - self.window + 1)]
        
        for window in windows:
            # 1. Drift: asimetría entre x y H_MAX-x
            drift = sum(abs(math.cos(math.pi * PHI * x) - 
                           math.cos(math.pi * PHI * (H_MAX - x))) 
                       for x in window) / (2 * len(window))
            
            # 2. Componente cuasiperiódica
            qp_mean = np.mean([((-1)**x) * math.cos(math.pi * PHI * x) for x in window])
            
            # 3. Balance antagonista
            antag_balance = np.mean([(H_MAX - x) in window for x in window])
            
            # 4. Torsión (GCD entre x y H_MAX-x)
            torsion_mean = np.mean([math.gcd(x, H_MAX - x) for x in window])
            
            features.append([drift, qp_mean, antag_balance, torsion_mean])
        
        return torch.tensor(features, dtype=torch.float32)  # (num_windows, 4)

    def forward(self, text: str, endian: str = "little") -> torch.Tensor:
        """Devuelve embedding H7 del texto"""
        # Convertir texto a secuencia Z7
        z7_seq = self.text_to_z7(text, endian)  # (L,)
        
        # Calcular características H7 por ventana
        features = self.compute_h7_features(z7_seq)  # (num_windows, 4)
        
        # Opcional: imprimir estadísticas
        if features.numel() > 0:
            features_np = features.cpu().numpy()
            if features_np.shape[1] >= 3:  # Verificar que antag_balance existe
                print("Antag balance stats:")
                print(f"  Mean: {features_np[:, 2].mean():.4f}")
                print(f"  Min:  {features_np[:, 2].min():.4f}") 
                print(f"  Max:  {features_np[:, 2].max():.4f}")
        
        # Proyectar características H7 a espacio de embedding
        embedding = self.proj(features)  # (num_windows, embed_dim)
        
        # Pooling sobre ventanas (mean + max)
        mean_pool = embedding.mean(dim=0)  # (embed_dim,)
        max_pool = embedding.max(dim=0).values  # (embed_dim,)
        
        # Concatenar resultados del pooling
        final_emb = torch.cat([mean_pool, max_pool])  # (2*embed_dim,)
        
        return final_emb.unsqueeze(0)  # (1, 2*embed_dim)

# ====================== USO ======================
if __name__ == "__main__":
    model = H7Embedding(embed_dim=64)
    
    texto = "Hola mundo, esto es una prueba de la métrica H7."
    
    emb_little = model(texto, endian="little")
    emb_big    = model(texto, endian="big")
        
    print(f"\nEmbedding Little shape: {emb_little.shape}")
    print(f"Embedding Big shape   : {emb_big.shape}")
    print(f"\nEjemplo embedding (primeros 8 valores):")
    print(emb_little[0][:8].tolist())
    
    # Comparación de embeddings
    similitud = torch.cosine_similarity(emb_little, emb_big).item()
    print(f"\nSimilitud coseno little vs big: {similitud:.4f}")


################################################################################
  IMPROVED MOD 6: EXACT BIG-ENDIAN HEXADECIMAL
################################################################################

|- COMPLETE HEXADECIMAL TABLE (Big-Endian)
|- unsigned 8-bit:   2 digits (00)
|- unsigned 16-bit:  4 digits (0000)
|- unsigned 32-bit:  8 digits (00000000)
|- unsigned 64-bit:  16 digits (0000000000000000)
|- unsigned 128-bit: 32 digits (00000000000000000000000000000000)

n | Index | Pair | Winding | Mapping | Weight | Discrete_phase | uint8 | uint16 | uint32   | uint64
--------------------------------------------------------------------------------------------------------------
0 |   1    |   6    |   0     |   0   |   1  |         0 |    28 |   0428 | 00000428 | 0000000000000428
1 |   5    |   2    |   0     |   0   |   1  |         1 |    0C |   0C0C | 00000C0C | 0000000000000C0C
2 |   3    |   4    |   0     |   0   |   1  |         6 |    1A |   341A | 0000341A | 0000000000

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Antag balance stats:
  Mean: 0.4674
  Min:  0.0000
  Max:  1.0000
Antag balance stats:
  Mean: 0.4674
  Min:  0.0000
  Max:  1.0000

Embedding Little shape: torch.Size([1, 128])
Embedding Big shape   : torch.Size([1, 128])

Ejemplo embedding (primeros 8 valores):
[-0.6659652590751648, -0.7316579222679138, -0.3808428645133972, 0.5408267974853516, 0.24924767017364502, 0.8653295636177063, -0.43854227662086487, 0.612080454826355]

Similitud coseno little vs big: 1.0000


In [6]:
"""
H7 FRAMEWORK - JSON/NDJSON DATA EXTRACTION PIPELINE
====================================================
Integrates covarianza decoder + metriplex oracle with data extraction
Handles both standard JSON and newline-delimited JSON (NDJSON) formats

Author: Jako (Jacobo Tlacaelel Mina Rodríguez)
Date: May 2026
"""

import os
import sys
import json
import pandas as pd
import numpy as np
import math
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# CONSTANTS & GLOBALS
# ============================================================================

phi = (1 + math.sqrt(5)) / 2
H_MAX = 7
O_n_integrity = 0.3624
DRIFT_072 = 7 - 2 * math.pi

# Golden ratio derived constants
PSI1 = abs(np.cos(np.pi * phi))
PSI2 = abs(np.sin(np.pi * phi))

# Data processing
TRAIN_RATIO = 0.70
TEST_RATIO = 0.20
BENCHMARK_RATIO = 0.10
RANDOM_SEED = 42

print(f"\n{'='*80}")
print(f"H7 Framework - JSON Extraction Pipeline Initialization")
print(f"{'='*80}")
print(f"[CONST] φ = {phi:.6f}")
print(f"[CONST] PSI1 (cos(πφ)) = {PSI1:.6f}")
print(f"[CONST] PSI2 (sin(πφ)) = {PSI2:.6f}")
print(f"[CONST] DRIFT_072 = {DRIFT_072:.6f}")


# ============================================================================
# PART 1: JSON FILE DISCOVERY & DETECTION
# ============================================================================

class JSONFileDiscovery:
    """Handles JSON file discovery and format detection."""
    
    @staticmethod
    def find_json_files(input_dir: str) -> List[str]:
        """Recursively finds all JSON files."""
        json_files = []
        for root, dirs, files in os.walk(input_dir):
            for file in files:
                if file.lower().endswith('.json'):
                    json_files.append(os.path.join(root, file))
        return sorted(json_files)
    
    @staticmethod
    def detect_json_format(json_path: str) -> str:
        """
        Detects if JSON file is standard JSON or NDJSON format.
        
        Returns:
            'standard' - Regular JSON (array or object)
            'ndjson' - Newline-delimited JSON (one object per line)
            'unknown' - Could not determine format
        """
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                # Try to read as standard JSON first
                content = f.read()
                if not content.strip():
                    return 'empty'
                
                try:
                    json.loads(content)
                    return 'standard'
                except json.JSONDecodeError:
                    pass
                
                # Check if it's NDJSON
                f.seek(0)
                lines = f.readlines()
                valid_ndjson = 0
                
                for line in lines[:min(10, len(lines))]:
                    line = line.strip()
                    if line:
                        try:
                            json.loads(line)
                            valid_ndjson += 1
                        except:
                            pass
                
                if valid_ndjson > 0:
                    return 'ndjson'
        
        except Exception as e:
            print(f"  [ERROR] Could not detect format: {e}")
            return 'unknown'
        
        return 'unknown'
    
    @staticmethod
    def get_file_size_mb(json_path: str) -> float:
        """Get file size in MB."""
        return os.path.getsize(json_path) / (1024 * 1024)


# ============================================================================
# PART 2: JSON EXTRACTION METHODS
# ============================================================================

class JSONExtractor:
    """Handles JSON data extraction with multiple format support."""
    
    @staticmethod
    def extract_ndjson(json_path: str) -> pd.DataFrame:
        """
        Extracts data from NDJSON (newline-delimited JSON) format.
        Each line is a separate JSON object.
        """
        records = []
        errors = 0
        line_count = 0
        
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f, 1):
                    line = line.strip()
                    
                    # Skip empty lines
                    if not line:
                        continue
                    
                    line_count += 1
                    
                    try:
                        record = json.loads(line)
                        records.append(record)
                    except json.JSONDecodeError as e:
                        errors += 1
                        if errors <= 3:
                            print(f"    [WARN] Line {line_num}: Invalid JSON - {str(e)[:40]}...")
        
        except Exception as e:
            print(f"    [ERROR] Reading file: {e}")
            return pd.DataFrame()
        
        if errors > 3:
            print(f"    [WARN] ... and {errors - 3} more JSON parse errors")
        
        if records:
            df = pd.DataFrame(records)
            print(f"    [OK] Extracted {len(records)} records from {line_count} lines")
            return df
        
        return pd.DataFrame()
    
    @staticmethod
    def extract_standard_json(json_path: str) -> pd.DataFrame:
        """Extracts data from standard JSON format (array or object)."""
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # If it's a list, treat as array of records
            if isinstance(data, list):
                df = pd.DataFrame(data)
                print(f"    [OK] Extracted {len(data)} records from JSON array")
                return df
            
            # If it's a dict, look for array keys
            elif isinstance(data, dict):
                # Common array keys
                for key in ['data', 'records', 'items', 'results', 'experiments', 
                           'rows', 'values', 'objects', 'entries', 'samples']:
                    if key in data and isinstance(data[key], list):
                        df = pd.DataFrame(data[key])
                        print(f"    [OK] Extracted {len(data[key])} records from '{key}' key")
                        return df
                
                # Try first array value found
                for key, value in data.items():
                    if isinstance(value, list) and len(value) > 0 and isinstance(value[0], dict):
                        df = pd.DataFrame(value)
                        print(f"    [OK] Extracted {len(value)} records from first array key '{key}'")
                        return df
                
                # Single record as dict
                df = pd.DataFrame([data])
                print(f"    [OK] Extracted 1 record from JSON object")
                return df
        
        except Exception as e:
            print(f"    [ERROR] Parsing standard JSON: {e}")
            return pd.DataFrame()
    
    @staticmethod
    def extract_json_auto(json_path: str) -> Optional[pd.DataFrame]:
        """
        Automatically detects JSON format and extracts data.
        """
        filename = os.path.basename(json_path)
        file_size = JSONFileDiscovery.get_file_size_mb(json_path)
        
        print(f"\n  📄 {filename} ({file_size:.2f} MB)")
        
        # Detect format
        format_type = JSONFileDiscovery.detect_json_format(json_path)
        print(f"    Format detected: {format_type}")
        
        if format_type == 'empty':
            print(f"    [SKIP] File is empty")
            return None
        
        # Extract based on format
        if format_type == 'ndjson':
            df = JSONExtractor.extract_ndjson(json_path)
        elif format_type == 'standard':
            df = JSONExtractor.extract_standard_json(json_path)
        else:
            # Try NDJSON first, then standard
            print(f"    Trying NDJSON format...")
            df = JSONExtractor.extract_ndjson(json_path)
            if df is None or len(df) == 0:
                print(f"    NDJSON failed, trying standard JSON...")
                df = JSONExtractor.extract_standard_json(json_path)
        
        return df if df is not None and len(df) > 0 else None


# ============================================================================
# PART 3: DATA CLEANING & FLATTENING
# ============================================================================

class DataProcessing:
    """Handles data cleaning and flattening."""
    
    @staticmethod
    def flatten_json_smart(df: pd.DataFrame) -> pd.DataFrame:
        """Intelligently flattens nested JSON structures."""
        df_flat = df.copy()
        columns_flattened = 0
        
        for col in df.columns:
            if df_flat[col].dtype == 'object':
                try:
                    sample = df_flat[col].dropna().iloc[0] if len(df_flat[col].dropna()) > 0 else None
                except:
                    continue
                
                if isinstance(sample, dict):
                    try:
                        flattened = df_flat[col].apply(lambda x: x if isinstance(x, dict) else {})
                        flattened_df = pd.json_normalize(flattened)
                        flattened_df.columns = [f'{col}_{sub_col}' for sub_col in flattened_df.columns]
                        df_flat = df_flat.drop(columns=[col])
                        df_flat = pd.concat([df_flat, flattened_df], axis=1)
                        columns_flattened += 1
                    except:
                        df_flat[col] = df_flat[col].apply(
                            lambda x: json.dumps(x) if isinstance(x, dict) else str(x)
                        )
                
                elif isinstance(sample, list):
                    df_flat[col] = df_flat[col].apply(
                        lambda x: json.dumps(x) if isinstance(x, list) else x
                    )
        
        print(f"    ✓ Flattened {columns_flattened} nested columns")
        return df_flat
    
    @staticmethod
    def clean_dataframe_safe(df: pd.DataFrame) -> pd.DataFrame:
        """Safely cleans data, handling unhashable types."""
        initial_shape = df.shape
        
        # Remove all-null columns
        df = df.dropna(axis=1, how='all')
        nullcols_removed = initial_shape[1] - df.shape[1]
        
        # Convert unhashable types to string BEFORE deduplication
        for col in df.columns:
            if df[col].dtype == 'object':
                try:
                    has_unhashable = False
                    for val in df[col].dropna():
                        if isinstance(val, (list, dict)):
                            has_unhashable = True
                            break
                    
                    if has_unhashable:
                        df[col] = df[col].apply(
                            lambda x: json.dumps(x) if isinstance(x, (list, dict)) 
                            else str(x) if pd.notna(x) else x
                        )
                except:
                    pass
        
        # Remove duplicates
        initial_rows = len(df)
        try:
            df = df.drop_duplicates()
            dupes_removed = initial_rows - len(df)
        except Exception as e:
            dupes_removed = 0
        
        # Try to convert to numeric
        for col in df.columns:
            if 'id' not in col.lower() and 'name' not in col.lower():
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    pass
        
        # Remove all-NaN rows
        initial_rows = len(df)
        df = df.dropna(how='all')
        nanrows_removed = initial_rows - len(df)
        
        df = df.reset_index(drop=True)
        
        print(f"    ✓ Removed {nullcols_removed} null columns, {dupes_removed} duplicates, {nanrows_removed} NaN rows")
        print(f"    ✓ Final shape: {df.shape}")
        
        return df


# ============================================================================
# PART 4: H7 FRAMEWORK ENRICHMENT
# ============================================================================

@dataclass
class H7Feature:
    """H7 Feature enrichment configuration."""
    name: str
    multiplier: float
    description: str


class H7Enrichment:
    """Adds H7 framework features to dataset."""
    
    H7_FEATURES = [
        H7Feature("phi_scaled", phi, "Golden ratio scaling"),
        H7Feature("psi1_scaled", PSI1, "cos(πφ) scaling"),
        H7Feature("psi2_scaled", PSI2, "sin(πφ) scaling"),
        H7Feature("drift_scaled", DRIFT_072, "DRIFT_072 scaling"),
        H7Feature("o_n_integrity_scaled", O_n_integrity, "O_n_integrity scaling"),
    ]
    
    @staticmethod
    def add_h7_features(df: pd.DataFrame) -> pd.DataFrame:
        """Adds H7 framework features to numeric columns."""
        df_enriched = df.copy()
        
        # Ensure ID column exists
        if 'id' not in df_enriched.columns:
            df_enriched.insert(0, 'id', range(len(df_enriched)))
        
        # Get numeric columns
        numeric_cols = df_enriched.select_dtypes(include=[np.number]).columns.tolist()
        
        # Skip ID column
        if 'id' in numeric_cols:
            numeric_cols.remove('id')
        
        features_added = 0
        
        # Add H7 features for each numeric column
        for col in numeric_cols[:20]:  # Limit to first 20 columns to avoid explosion
            try:
                for h7_feat in H7Enrichment.H7_FEATURES:
                    new_col_name = f'{col}_{h7_feat.name}'
                    df_enriched[new_col_name] = df_enriched[col] * h7_feat.multiplier
                    features_added += 1
            except:
                pass
        
        print(f"    ✓ Added {features_added} H7 enrichment features")
        return df_enriched
    
    @staticmethod
    def add_h7_statistics(df: pd.DataFrame) -> pd.DataFrame:
        """Adds H7-derived statistical features."""
        df_stats = df.copy()
        
        numeric_cols = df_stats.select_dtypes(include=[np.number]).columns.tolist()
        if 'id' in numeric_cols:
            numeric_cols.remove('id')
        
        if not numeric_cols:
            return df_stats
        
        # Add statistical features
        try:
            # Row-wise statistics
            df_stats['row_mean'] = df_stats[numeric_cols].mean(axis=1)
            df_stats['row_std'] = df_stats[numeric_cols].std(axis=1)
            df_stats['row_max'] = df_stats[numeric_cols].max(axis=1)
            df_stats['row_min'] = df_stats[numeric_cols].min(axis=1)
            
            # H7-weighted mean
            df_stats['row_h7_weighted_mean'] = df_stats[numeric_cols].mul(phi).mean(axis=1)
            
            print(f"    ✓ Added 5 statistical features")
        except:
            print(f"    [WARN] Could not add statistical features")
        
        return df_stats


# ============================================================================
# PART 5: DATA SPLITTING
# ============================================================================

class DataSplitting:
    """Handles train/test/benchmark splitting."""
    
    @staticmethod
    def create_data_splits(
        df: pd.DataFrame,
        train_ratio: float = TRAIN_RATIO,
        test_ratio: float = TEST_RATIO,
        benchmark_ratio: float = BENCHMARK_RATIO,
        random_seed: int = RANDOM_SEED
    ) -> Dict[str, pd.DataFrame]:
        """Splits data into train, test, and benchmark sets."""
        
        np.random.seed(random_seed)
        
        n_samples = len(df)
        indices = np.arange(n_samples)
        np.random.shuffle(indices)
        
        n_train = int(n_samples * train_ratio)
        n_test = int(n_samples * test_ratio)
        
        train_idx = indices[:n_train]
        test_idx = indices[n_train:n_train + n_test]
        benchmark_idx = indices[n_train + n_test:]
        
        splits = {
            'train': df.iloc[train_idx].reset_index(drop=True),
            'test': df.iloc[test_idx].reset_index(drop=True),
            'benchmark': df.iloc[benchmark_idx].reset_index(drop=True)
        }
        
        return splits


# ============================================================================
# PART 6: MAIN PIPELINE
# ============================================================================

class H7ExtractionPipeline:
    """Main data extraction pipeline."""
    
    def __init__(self, input_dir: str = "/kaggle/input", output_dir: str = "/kaggle/working"):
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.splits = None
        self.summary = None
    
    def run(self) -> Tuple[Dict[str, pd.DataFrame], Dict]:
        """Execute the full pipeline."""
        
        print(f"\n{'='*80}")
        print("H7 FRAMEWORK - JSON EXTRACTION PIPELINE")
        print(f"{'='*80}")
        
        # STEP 1: Find files
        print(f"\n[STEP 1] Discovering JSON files...")
        print(f"{'-'*80}")
        
        json_files = JSONFileDiscovery.find_json_files(self.input_dir)
        print(f"Found {len(json_files)} JSON file(s)")
        
        if not json_files:
            print("[ERROR] No JSON files found")
            return {}, {}
        
        for i, jf in enumerate(json_files, 1):
            print(f"  {i}. {os.path.basename(jf)}")
        
        # STEP 2: Extract data
        print(f"\n[STEP 2] Extracting data from JSON files...")
        print(f"{'-'*80}")
        
        all_extracted_data = []
        
        for json_file in json_files:
            df = JSONExtractor.extract_json_auto(json_file)
            if df is not None and len(df) > 0:
                all_extracted_data.append(df)
        
        if not all_extracted_data:
            print("[ERROR] Could not extract data from any JSON files")
            return {}, {}
        
        # STEP 3: Consolidate
        print(f"\n[STEP 3] Consolidating data...")
        print(f"{'-'*80}")
        
        full_dataset = pd.concat(all_extracted_data, ignore_index=True, sort=False)
        print(f"  ✓ Consolidated: {full_dataset.shape[0]} rows × {full_dataset.shape[1]} columns")
        
        # STEP 4: Flatten
        print(f"\n[STEP 4] Flattening nested structures...")
        print(f"{'-'*80}")
        
        full_dataset = DataProcessing.flatten_json_smart(full_dataset)
        print(f"  ✓ Flattened: {full_dataset.shape[0]} rows × {full_dataset.shape[1]} columns")
        
        # STEP 5: Clean
        print(f"\n[STEP 5] Cleaning data...")
        print(f"{'-'*80}")
        
        full_dataset = DataProcessing.clean_dataframe_safe(full_dataset)
        
        # STEP 6: Enrich with H7
        print(f"\n[STEP 6] Enriching with H7 framework...")
        print(f"{'-'*80}")
        
        full_dataset = H7Enrichment.add_h7_features(full_dataset)
        full_dataset = H7Enrichment.add_h7_statistics(full_dataset)
        print(f"  ✓ Enriched: {full_dataset.shape[0]} rows × {full_dataset.shape[1]} columns")
        
        # STEP 7: Split
        print(f"\n[STEP 7] Creating train/test/benchmark splits...")
        print(f"{'-'*80}")
        
        self.splits = DataSplitting.create_data_splits(full_dataset)
        
        print(f"  ✓ Train:     {self.splits['train'].shape[0]} samples ({TRAIN_RATIO*100:.0f}%)")
        print(f"  ✓ Test:      {self.splits['test'].shape[0]} samples ({TEST_RATIO*100:.0f}%)")
        print(f"  ✓ Benchmark: {self.splits['benchmark'].shape[0]} samples ({BENCHMARK_RATIO*100:.0f}%)")
        
        # STEP 8: Save
        print(f"\n[STEP 8] Saving splits...")
        print(f"{'-'*80}")
        
        os.makedirs(self.output_dir, exist_ok=True)
        
        self.splits['train'].to_csv(os.path.join(self.output_dir, 'train.csv'), index=False)
        self.splits['test'].to_csv(os.path.join(self.output_dir, 'test.csv'), index=False)
        self.splits['benchmark'].to_csv(os.path.join(self.output_dir, 'benchmark.csv'), index=False)
        
        print(f"  ✓ train.csv ({self.splits['train'].shape[0]} rows)")
        print(f"  ✓ test.csv ({self.splits['test'].shape[0]} rows)")
        print(f"  ✓ benchmark.csv ({self.splits['benchmark'].shape[0]} rows)")
        
        # STEP 9: Summary
        print(f"\n[STEP 9] Generating summary...")
        print(f"{'-'*80}")
        
        self.summary = {
            'pipeline_timestamp': pd.Timestamp.now().isoformat(),
            'input_json_files': len(json_files),
            'total_samples': len(full_dataset),
            'total_features': full_dataset.shape[1],
            'train_samples': self.splits['train'].shape[0],
            'test_samples': self.splits['test'].shape[0],
            'benchmark_samples': self.splits['benchmark'].shape[0],
            'h7_constants': {
                'phi': float(phi),
                'psi1': float(PSI1),
                'psi2': float(PSI2),
                'drift_072': float(DRIFT_072),
                'o_n_integrity': float(O_n_integrity),
            }
        }
        
        with open(os.path.join(self.output_dir, 'pipeline_summary.json'), 'w') as f:
            json.dump(self.summary, f, indent=2)
        
        print(f"  ✓ Summary saved to pipeline_summary.json")
        
        print(f"\n{'='*80}")
        print("✓ PIPELINE COMPLETED SUCCESSFULLY")
        print(f"{'='*80}\n")
        
        return self.splits, self.summary


# ============================================================================
# USAGE EXAMPLES & ENTRY POINT
# ============================================================================

def example_standalone_usage():
    """Example: Using the pipeline standalone."""
    print("\n[EXAMPLE 1] Standalone pipeline usage")
    print("-" * 80)
    
    pipeline = H7ExtractionPipeline(
        input_dir="/kaggle/input",
        output_dir="/kaggle/working"
    )
    
    splits, summary = pipeline.run()
    
    return splits, summary


def example_with_custom_processing():
    """Example: Custom processing after extraction."""
    print("\n[EXAMPLE 2] Custom processing after extraction")
    print("-" * 80)
    
    pipeline = H7ExtractionPipeline()
    splits, summary = pipeline.run()
    
    # Custom processing
    if splits:
        train_df = splits['train']
        
        # Example: Filter by H7 enriched column
        if 'id_phi_scaled' in train_df.columns:
            high_phi_samples = train_df[train_df['id_phi_scaled'] > phi]
            print(f"  Samples with phi-scaled ID > φ: {len(high_phi_samples)}")
    
    return splits, summary


if __name__ == '__main__':
    
    # Check if running in Kaggle environment
    is_kaggle = os.path.exists("/kaggle/working")
    
    if is_kaggle:
        print("✓ Running in Kaggle environment")
        splits, summary = example_standalone_usage()
    else:
        print("⚠ Not in Kaggle environment (testing mode)")
        print("  Using default paths (will create test structure)")
        
        # Create test directory structure
        os.makedirs("/kaggle/input/", exist_ok=True)
        os.makedirs("/kaggle/output/", exist_ok=True)
        
        # Create test JSON file (NDJSON format)
        test_ndjson = "/kaggle/input/notebooks/jakomina/h-logic/test.csv"
        with open(test_ndjson, 'w') as f:
            for i in range(100):
                record = {
                    'id': i,
                    'value': np.random.randn(),
                    'category': np.random.choice(['A', 'B', 'C']),
                    'metrics': {
                        'score': np.random.random(),
                        'count': np.random.randint(0, 100)
                    }
                }
                f.write(json.dumps(record) + '\n')
        
        print(f"✓ Created test NDJSON file: {test_ndjson}")
        
        # Run pipeline
        pipeline = H7ExtractionPipeline(
            input_dir="/kaggle/input/notebooks/jakomina/h-logic/train.csv",
            output_dir="/kaggle/working/"
        )
        splits, summary = pipeline.run()
        
        # Show results
        if splits:
            print("\n[TEST RESULTS]")
            print(f"Train data sample:")
            print(splits['train'].head())
    
    print("\n✓ H7 JSON Extraction Pipeline loaded successfully")


H7 Framework - JSON Extraction Pipeline Initialization
[CONST] φ = 1.618034
[CONST] PSI1 (cos(πφ)) = 0.362375
[CONST] PSI2 (sin(πφ)) = 0.932032
[CONST] DRIFT_072 = 0.716815
✓ Running in Kaggle environment

[EXAMPLE 1] Standalone pipeline usage
--------------------------------------------------------------------------------

H7 FRAMEWORK - JSON EXTRACTION PIPELINE

[STEP 1] Discovering JSON files...
--------------------------------------------------------------------------------
Found 5 JSON file(s)
  1. datos_experimento1.json
  2. __huggingface_repos__.json
  3. __output__.json
  4. h7_unified_report.json
  5. pipeline_summary.json

[STEP 2] Extracting data from JSON files...
--------------------------------------------------------------------------------

  📄 datos_experimento1.json (26.01 MB)
    Format detected: standard
    [OK] Extracted 1 record from JSON object

  📄 __huggingface_repos__.json (0.00 MB)
    Format detected: standard
    [OK] Extracted 1 records from first array